
# Milestone 07 — Calibration, Business Costs & Threshold Selection

**Project:** Bank Account Fraud Detection — Deep Learning & MLOps  
**Status:** In Progress  
**Primary objective:** Develop and evaluate probability calibration and operational fraud-detection policies without contaminating the sealed final test set.

## 1. Learning Objectives

By the end of this milestone, we will:

1. Build leakage-safe calibration-development datasets.
2. Refit preprocessing using training data only.
3. Train development versions of HistGradientBoosting and the weighted PyTorch MLP.
4. Compare uncalibrated, sigmoid-calibrated, and isotonic-calibrated predictions.
5. Evaluate Brier score, log loss, PR-AUC, and reliability diagrams.
6. Model fraud-related business costs and investigation capacity.
7. Compare operational thresholds and alert budgets.
8. Document a defensible provisional model and operating policy.

## 2. Chronological Data Design

| Partition | Months | Applications | Fraud Cases |
|---|---|---:|---:|
| Training | 0–4 | 675,666 | 6,740 |
| Calibration | 5 | 119,323 | 1,411 |
| Exploratory evaluation | 6 | 108,168 | 1,450 |
| Final holdout | 7 | SEALED | Not accessed |

### Leakage Prevention Rules

- Fit preprocessing only on training months 0–4.
- Fit calibration-development models only on months 0–4.
- Fit probability calibrators only on month 5.
- Use month 6 for exploratory evaluation and policy development.
- Never use month 7 for training, calibration, or threshold selection.
- Preserve all Milestone 06 models and experiment records.

**Important:** Models trained on months 0–4 are separate calibration-development models. Calibration results cannot automatically be transferred to the original Milestone 06 models trained on months 0–5.

Month 6 has already been used for model selection, so its findings are exploratory rather than independent confirmation.

## 3. Project Resources

**Development dataset:** `artifacts/milestone_07/development_months_0_to_6.csv`

**Split configuration:** `configs/milestone_07_split_config.json`

**Preprocessing module:** `src/features/preprocessing.py`

**PyTorch model:** `src/models/mlp.py`

**Training functions:** `src/training/loops.py`

**Evaluation functions:** `src/evaluation/metrics.py`

## 4. Execution Policy

Execute notebook cells sequentially and verify each checkpoint before proceeding.

Save intermediate reports and model artifacts under `artifacts/milestone_07/`, which is excluded from Git.

Do not save raw customer records or trained model weights in the Git repository.

**Final holdout status: MONTH 7 SEALED.**


### Environment and Configuration Verification

In [ ]:

# ============================================================
# Milestone 07 — Cell 2
# Environment, Configuration, and Data Safety Verification
# ============================================================

from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
import sklearn
import torch

# 1. Locate the project root.
current = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in (current, *current.parents)
        if (
            path / "configs/milestone_07_split_config.json"
        ).is_file()
    ),
    None,
)

assert PROJECT_ROOT is not None, "Project root not found"

# 2. Confirm the correct Python environment.
EXPECTED_ENV = (PROJECT_ROOT / ".venv").resolve()
ACTIVE_ENV = Path(sys.prefix).resolve()

assert ACTIVE_ENV == EXPECTED_ENV, (
    f"Wrong Jupyter kernel!\n"
    f"Active: {ACTIVE_ENV}\n"
    f"Expected: {EXPECTED_ENV}"
)

# 3. Confirm the correct Git branch.
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=PROJECT_ROOT,
    text=True,
).strip()

assert branch == "milestone-07-calibration-business-costs", (
    f"Wrong Git branch: {branch}"
)

# 4. Load and verify calibration configuration.
CONFIG_PATH = (
    PROJECT_ROOT / "configs/milestone_07_split_config.json"
)

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    split_config = json.load(file)

expected_splits = {
    "train": [0, 1, 2, 3, 4],
    "calibration": [5],
    "evaluation": [6],
    "holdout": [7],
}

for name, months in expected_splits.items():
    assert split_config[name]["months"] == months

assert split_config["holdout"]["status"] == "SEALED"
assert split_config["holdout"]["access_allowed"] is False
assert split_config["safeguards"]["prohibit_holdout_access"] is True

# 5. Locate the protected development dataset.
DATA_PATH = (
    PROJECT_ROOT
    / "artifacts/milestone_07/development_months_0_to_6.csv"
)

assert DATA_PATH.is_file(), "Development snapshot missing"

# 6. Make project modules importable.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 7. Print the verification report.
print("MILESTONE 07 — NOTEBOOK ENVIRONMENT CHECK")
print("=" * 55)

print("Project root:", PROJECT_ROOT)
print("Git branch:", branch)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("\nChronological partitions:")
for name, months in expected_splits.items():
    print(f"  {name}: {months}")

print("\nDevelopment dataset: FOUND")
print("Correct Jupyter kernel: PASS")
print("Month-7 holdout: SEALED")
print("Dataset rows loaded: 0")

print("\nStep 07.26: PASS")


### Load and Separate Development Data

In [ ]:

# ============================================================
# Milestone 07 — Cell 3
# Load and Separate Protected Development Data
# ============================================================

from src.features.preprocessing import (
    load_preprocessing_config,
)

# 1. Load our approved feature configuration.
feature_config = load_preprocessing_config()

numeric_features = feature_config["numeric_features"]
categorical_features = feature_config["categorical_features"]
binary_features = feature_config["binary_features"]

model_features = (
    numeric_features
    + categorical_features
    + binary_features
)

# 2. Verify that excluded columns are not model features.
excluded_columns = {
    "fraud_bool",
    "month",
    "device_fraud_count",
}

assert excluded_columns.isdisjoint(model_features), (
    "Data leakage: excluded feature detected"
)

assert len(model_features) == len(set(model_features)), (
    "Duplicate model features detected"
)

# 3. Load ONLY the protected development snapshot.
# Reduce memory usage by specifying appropriate dtypes.
column_types = {
    "month": "int8",
    "fraud_bool": "int8",
}

column_types.update({
    col: "float32"
    for col in numeric_features + binary_features
})

column_types.update({
    col: "category"
    for col in categorical_features
})

required_columns = [
    "month",
    "fraud_bool",
] + model_features

development_df = pd.read_csv(
    DATA_PATH,
    usecols=required_columns,
    dtype=column_types,
)

# 4. Verify the snapshot's boundaries and labels.
assert len(development_df) == 903157, (
    "Unexpected development dataset size"
)

assert set(development_df["month"].unique()) == set(range(7)), (
    "Unexpected or missing development months"
)

assert development_df["fraud_bool"].notna().all(), (
    "Missing target labels detected"
)

assert development_df["fraud_bool"].isin([0, 1]).all(), (
    "Invalid binary target values"
)

assert 7 not in development_df["month"].unique(), (
    "SEALED HOLDOUT LEAKAGE DETECTED"
)

# 5. Create separate chronological partitions.
train_df = development_df.loc[
    development_df["month"].isin(
        split_config["train"]["months"]
    )
].copy()

calibration_df = development_df.loc[
    development_df["month"].isin(
        split_config["calibration"]["months"]
    )
].copy()

evaluation_df = development_df.loc[
    development_df["month"].isin(
        split_config["evaluation"]["months"]
    )
].copy()

# 6. Verify row counts and fraud counts.
expected_counts = {
    "Training": (675666, 6740),
    "Calibration": (119323, 1411),
    "Evaluation": (108168, 1450),
}

partitions = {
    "Training": train_df,
    "Calibration": calibration_df,
    "Evaluation": evaluation_df,
}

for name, frame in partitions.items():
    expected_rows, expected_fraud = expected_counts[name]

    assert len(frame) == expected_rows, (
        f"{name} row count mismatch"
    )

    assert int(frame["fraud_bool"].sum()) == expected_fraud, (
        f"{name} fraud count mismatch"
    )

assert sum(len(frame) for frame in partitions.values()) == len(
    development_df
), "Dataset partition mismatch"

# 7. Release the combined DataFrame to conserve memory.
del development_df

# 8. Display aggregate verification results.
print("MILESTONE 07 — CHRONOLOGICAL DATA PARTITIONS")
print("=" * 65)

print(
    f"{'Dataset':<15}"
    f"{'Applications':>15}"
    f"{'Fraud Cases':>15}"
    f"{'Fraud Rate':>15}"
)

for name, frame in partitions.items():
    applications = len(frame)
    fraud_cases = int(frame["fraud_bool"].sum())
    fraud_rate = fraud_cases / applications

    print(
        f"{name:<15}"
        f"{applications:>15,}"
        f"{fraud_cases:>15,}"
        f"{fraud_rate:>14.3%}"
    )

print("\nOriginal input features:", len(model_features))
print("Excluded leakage columns: VERIFIED")
print("Chronological separation: PASS")
print("Month-7 records loaded: 0")

print("\nStep 07.27: PASS")


### Leakage-Safe Feature Preprocessing

In [ ]:

# ============================================================
# Milestone 07 — Cell 4
# Leakage-Safe Feature Preprocessing
# ============================================================

from src.features.preprocessing import (
    prepare_model_features,
    build_preprocessor,
)

# 1. Build a NEW unfitted preprocessor.
# Never reuse the M05/M06 preprocessor fitted on months 0-5.
preprocessor = build_preprocessor(config=feature_config)

# 2. Fit preprocessing EXCLUSIVELY on months 0-4.
train_features = prepare_model_features(
    train_df,
    config=feature_config,
)

X_train = np.asarray(
    preprocessor.fit_transform(train_features),
    dtype=np.float32,
)

del train_features

# 3. Transform month 5 WITHOUT refitting.
calibration_features = prepare_model_features(
    calibration_df,
    config=feature_config,
)

X_calibration = np.asarray(
    preprocessor.transform(calibration_features),
    dtype=np.float32,
)

del calibration_features

# 4. Transform month 6 WITHOUT refitting.
evaluation_features = prepare_model_features(
    evaluation_df,
    config=feature_config,
)

X_evaluation = np.asarray(
    preprocessor.transform(evaluation_features),
    dtype=np.float32,
)

del evaluation_features

# 5. Extract binary classification targets.
y_train = train_df["fraud_bool"].to_numpy(
    dtype=np.float32,
    copy=True,
)

y_calibration = calibration_df["fraud_bool"].to_numpy(
    dtype=np.float32,
    copy=True,
)

y_evaluation = evaluation_df["fraud_bool"].to_numpy(
    dtype=np.float32,
    copy=True,
)

# 6. Retrieve transformed feature names.
feature_names = preprocessor.get_feature_names_out()

# 7. Verify dimensions, data types, and values.
datasets = {
    "Training": (X_train, y_train, 675666, 6740),
    "Calibration": (
        X_calibration,
        y_calibration,
        119323,
        1411,
    ),
    "Evaluation": (
        X_evaluation,
        y_evaluation,
        108168,
        1450,
    ),
}

for name, (X, y, expected_rows, expected_fraud) in datasets.items():

    assert X.shape == (expected_rows, 55), (
        f"{name}: unexpected feature shape {X.shape}"
    )

    assert y.shape == (expected_rows,), (
        f"{name}: unexpected target shape"
    )

    assert X.dtype == np.float32, (
        f"{name}: features must be float32"
    )

    assert y.dtype == np.float32, (
        f"{name}: targets must be float32"
    )

    assert np.isfinite(X).all(), (
        f"{name}: non-finite feature values detected"
    )

    assert np.isin(y, [0, 1]).all(), (
        f"{name}: invalid target values"
    )

    assert int(y.sum()) == expected_fraud, (
        f"{name}: fraud count mismatch"
    )

assert len(feature_names) == 55
assert len(set(feature_names)) == 55

# 8. Display verification results.
print("MILESTONE 07 — PREPROCESSING VERIFICATION")
print("=" * 60)

for name, (X, y, _, _) in datasets.items():
    print(
        f"{name:<15}"
        f"X: {str(X.shape):<18}"
        f"y: {str(y.shape):<12}"
        f"dtype: {X.dtype}"
    )

print("\nTransformed features:", len(feature_names))
print("Preprocessor fitted on months 0-4: YES")
print("Calibration data used for fitting: NO")
print("Evaluation data used for fitting: NO")
print("Finite values: PASS")
print("Month-7 records accessed: 0")

print("\nStep 07.28: PASS")


### Train the HistGradientBoosting Calibration-Development Model

In [ ]:

# ============================================================
# Milestone 07 — Cell 5
# Train HistGradientBoosting Calibration-Development Model
# ============================================================

import json
import time

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
)

# 1. Retrieve the verified Milestone 06 HGB hyperparameters.
reference_path = (
    PROJECT_ROOT
    / "artifacts/milestone_06/m06_hgb_reproduction_v01.json"
)

with reference_path.open("r", encoding="utf-8") as file:
    hgb_reference = json.load(file)

hgb_params = hgb_reference["parameters"]

assert hgb_params["max_iter"] == 100
assert hgb_params["learning_rate"] == 0.1
assert hgb_params["max_leaf_nodes"] == 31
assert hgb_params["early_stopping"] is False
assert hgb_params["random_state"] == 42

# 2. Initialize a NEW model.
# Do not overwrite the original Milestone 06 HGB model.
hgb_dev = HistGradientBoostingClassifier(**hgb_params)

# 3. Train ONLY on months 0-4.
start_time = time.perf_counter()

hgb_dev.fit(
    X_train,
    y_train.astype(np.int8),
)

hgb_training_seconds = time.perf_counter() - start_time

# 4. Generate out-of-sample predictions for month 5.
hgb_calibration_raw = hgb_dev.predict_proba(
    X_calibration
)[:, 1]

# 5. Generate exploratory predictions for month 6.
hgb_evaluation_raw = hgb_dev.predict_proba(
    X_evaluation
)[:, 1]

# 6. Validate predictions and fitted model.
for name, predictions, labels in [
    ("Calibration", hgb_calibration_raw, y_calibration),
    ("Evaluation", hgb_evaluation_raw, y_evaluation),
]:
    assert predictions.shape == labels.shape, (
        f"{name}: prediction shape mismatch"
    )

    assert np.isfinite(predictions).all(), (
        f"{name}: non-finite predictions"
    )

    assert np.all(
        (predictions >= 0) & (predictions <= 1)
    ), f"{name}: predictions outside [0, 1]"

assert hgb_dev.n_iter_ == 100, (
    "Unexpected number of boosting iterations"
)

# 7. Report uncalibrated performance.
print("MILESTONE 07 — HGB DEVELOPMENT MODEL")
print("=" * 60)
print("Training months: 0-4")
print("Training records:", f"{len(y_train):,}")
print("Model iterations:", hgb_dev.n_iter_)
print("Training time:", f"{hgb_training_seconds:.2f} seconds")

print("\nUNCALIBRATED PERFORMANCE")
print("-" * 60)

for name, predictions, labels in [
    ("Calibration (month 5)", hgb_calibration_raw, y_calibration),
    ("Evaluation (month 6)", hgb_evaluation_raw, y_evaluation),
]:
    print(f"\n{name}")
    print(
        "  PR-AUC:",
        f"{average_precision_score(labels, predictions):.6f}",
    )
    print(
        "  Brier score:",
        f"{brier_score_loss(labels, predictions):.6f}",
    )
    print(
        "  Log loss:",
        f"{log_loss(labels, predictions):.6f}",
    )

print("\nPreprocessor fit: Months 0-4 only")
print("HGB training: Months 0-4 only")
print("Calibration model fitted: NO")
print("Month-7 records accessed: 0")
print("\nStep 07.29: PASS")


## HGB Summary
**Two observations matter:**
- Ranking performance declines in month 6. This could reflect changing fraud patterns, shifts in the applicant population, or ordinary variation.
- Probability prediction quality also declines. Both Brier score and log loss increase in month 6.

The Milestone 06 HGB achieved a higher month-6 PR-AUC of 0.166961, but it was trained on months 0–5. This new model was trained only on months 0–4, so the results are not directly interchangeable.

### Sigmoid (Platt) Calibration
Platt scaling uses logistic regression to transform a model's scores into better-calibrated probabilities.
Instead of changing the underlying HGB model, we learn a mapping:
\[
P(\text{fraud})=\frac{1}{1+e^{-(as+b)}}
\]
where \(s\) is the original model's log-odds score, and \(a\) and \(b\) are learned from month 5.
Because month 5 was excluded from HGB training, these are out-of-sample scores.

In [ ]:

# ============================================================
# Milestone 07 — Cell 6
# Sigmoid (Platt) Calibration for HistGradientBoosting
# ============================================================

from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
)

# 1. Convert probabilities to finite log-odds scores.
# Clipping avoids logit(0) and logit(1).
EPSILON = 1e-7

def probability_to_logit(probabilities):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    clipped = np.clip(
        probabilities,
        EPSILON,
        1.0 - EPSILON,
    )

    return logit(clipped).reshape(-1, 1)


# 2. Prepare calibration-month scores.
X_platt_calibration = probability_to_logit(
    hgb_calibration_raw
)

# 3. Fit calibration ONLY using month 5.
hgb_platt = LogisticRegression(
    C=1e6,
    solver="lbfgs",
    max_iter=1000,
    random_state=42,
)

hgb_platt.fit(
    X_platt_calibration,
    y_calibration.astype(np.int8),
)

# 4. Transform month-5 predictions for diagnostics.
# These are IN-SAMPLE calibration diagnostics.
hgb_calibration_platt = hgb_platt.predict_proba(
    X_platt_calibration
)[:, 1]

# 5. Transform month-6 predictions WITHOUT refitting.
X_platt_evaluation = probability_to_logit(
    hgb_evaluation_raw
)

hgb_evaluation_platt = hgb_platt.predict_proba(
    X_platt_evaluation
)[:, 1]

# 6. Validate the calibrated probabilities.
for name, predictions, labels in [
    ("Calibration", hgb_calibration_platt, y_calibration),
    ("Evaluation", hgb_evaluation_platt, y_evaluation),
]:
    assert predictions.shape == labels.shape, (
        f"{name}: shape mismatch"
    )

    assert np.isfinite(predictions).all(), (
        f"{name}: non-finite predictions"
    )

    assert np.all(
        (predictions >= 0) & (predictions <= 1)
    ), f"{name}: invalid probability range"

assert hgb_platt.coef_[0, 0] > 0, (
    "Unexpected negative calibration slope"
)

# 7. Compare original and calibrated performance.
print("MILESTONE 07 — HGB SIGMOID CALIBRATION")
print("=" * 65)

print("Base model training: Months 0-4")
print("Calibrator fitted on: Month 5")
print("Calibration method: Logistic regression on log-odds")
print("Calibration slope:", round(hgb_platt.coef_[0, 0], 6))
print("Calibration intercept:", round(hgb_platt.intercept_[0], 6))

print("\nMONTH 6 — EXPLORATORY EVALUATION")
print("-" * 65)

results = []

for name, predictions in [
    ("Uncalibrated HGB", hgb_evaluation_raw),
    ("Sigmoid-calibrated HGB", hgb_evaluation_platt),
]:
    results.append({
        "Model": name,
        "PR-AUC": average_precision_score(
            y_evaluation, predictions
        ),
        "Brier": brier_score_loss(
            y_evaluation, predictions
        ),
        "Log Loss": log_loss(
            y_evaluation, predictions
        ),
        "Mean Predicted": float(np.mean(predictions)),
    })

calibration_comparison = pd.DataFrame(results)

display(
    calibration_comparison.round(6)
)

print(
    "\nActual month-6 fraud rate:",
    f"{np.mean(y_evaluation):.6f}",
)

print(
    "Brier improvement (positive = better):",
    f"{results[0]['Brier'] - results[1]['Brier']:.8f}",
)

print(
    "Log-loss improvement (positive = better):",
    f"{results[0]['Log Loss'] - results[1]['Log Loss']:.8f}",
)

print("\nMonth 5 used for calibration: YES")
print("Month 6 used to fit calibrator: NO")
print("Month 6 evaluation status: EXPLORATORY")
print("Month 7 accessed: NO")

print("\nStep 07.30: PASS")


## Sigmoud Summary
The actual fraud rate in month 6 was 1.3405%, so sigmoid calibration brought the average predicted fraud probability closer to the observed rate.
However, a better average probability does not automatically mean better individual probability predictions. That's why we examine multiple calibration metrics.

### Isotonic Calibration
Isotonic regression learns a flexible, non-decreasing relationship between model scores and observed fraud rates.
Unlike sigmoid calibration, it doesn't assume the relationship follows a logistic curve

In [ ]:

# ============================================================
# Milestone 07 — Cell 7
# Isotonic Calibration for HistGradientBoosting
# ============================================================

from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
)

# 1. Create a new isotonic calibrator.
# The original HGB model remains unchanged.
hgb_isotonic = IsotonicRegression(
    y_min=0.0,
    y_max=1.0,
    increasing=True,
    out_of_bounds="clip",
)

# 2. Fit ONLY on month-5 out-of-sample predictions.
hgb_isotonic.fit(
    hgb_calibration_raw,
    y_calibration.astype(np.int8),
)

# 3. Generate fitted calibration-month probabilities.
# These are IN-SAMPLE calibration diagnostics.
hgb_calibration_isotonic = hgb_isotonic.predict(
    hgb_calibration_raw
)

# 4. Apply the fitted calibrator to month 6.
# Do not refit using evaluation labels.
hgb_evaluation_isotonic = hgb_isotonic.predict(
    hgb_evaluation_raw
)

# 5. Verify output integrity.
for name, predictions, labels in [
    ("Calibration", hgb_calibration_isotonic, y_calibration),
    ("Evaluation", hgb_evaluation_isotonic, y_evaluation),
]:
    assert predictions.shape == labels.shape, (
        f"{name}: shape mismatch"
    )

    assert np.isfinite(predictions).all(), (
        f"{name}: non-finite predictions"
    )

    assert np.all(
        (predictions >= 0) & (predictions <= 1)
    ), f"{name}: invalid probability range"

# 6. Compare all three approaches on month 6.
methods = {
    "Uncalibrated HGB": hgb_evaluation_raw,
    "Sigmoid HGB": hgb_evaluation_platt,
    "Isotonic HGB": hgb_evaluation_isotonic,
}

comparison_rows = []

actual_fraud_rate = float(np.mean(y_evaluation))

for name, predictions in methods.items():

    comparison_rows.append({
        "Method": name,
        "PR-AUC": average_precision_score(
            y_evaluation,
            predictions,
        ),
        "Brier": brier_score_loss(
            y_evaluation,
            predictions,
        ),
        "Log Loss": log_loss(
            y_evaluation,
            predictions,
            labels=[0, 1],
        ),
        "Mean Predicted": float(
            np.mean(predictions)
        ),
        "Mean Rate Error": abs(
            float(np.mean(predictions))
            - actual_fraud_rate
        ),
    })

hgb_calibration_results = pd.DataFrame(
    comparison_rows
)

# 7. Summarize results.
print("MILESTONE 07 — HGB CALIBRATION COMPARISON")
print("=" * 65)

print("Base model trained: Months 0-4")
print("Calibrators fitted: Month 5")
print("Exploratory evaluation: Month 6")

print("\nMONTH 6 — THREE-METHOD COMPARISON")
print("-" * 65)

display(
    hgb_calibration_results.round(6)
)

print(
    "\nActual fraud rate:",
    f"{actual_fraud_rate:.6f}",
)

# 8. Identify exploratory metric leaders.
best_brier = hgb_calibration_results.loc[
    hgb_calibration_results["Brier"].idxmin(),
    "Method",
]

best_log_loss = hgb_calibration_results.loc[
    hgb_calibration_results["Log Loss"].idxmin(),
    "Method",
]

print("\nLowest Brier score:", best_brier)
print("Lowest log loss:", best_log_loss)

# 9. Calibration complexity diagnostics.
print(
    "\nIsotonic fitted thresholds:",
    len(hgb_isotonic.X_thresholds_),
)

print(
    "Unique month-6 isotonic probabilities:",
    len(np.unique(hgb_evaluation_isotonic)),
)

print("\nMonth 5 used to fit calibrators: YES")
print("Month 6 used to fit calibrators: NO")
print("Evaluation status: EXPLORATORY")
print("Month 7 accessed: NO")

print("\nStep 07.31: PASS")


## Isotonic Summary
The actual month-6 fraud rate was 1.3405%.

**What does this mean?**
- Uncalibrated HGB: Best ranking performance, tied with sigmoid.
- Sigmoid HGB: Improves log loss while preserving PR-AUC, but slightly worsens Brier score.
- Isotonic HGB: Best Brier score and log loss, but PR-AUC falls from 0.144497 to 0.134553.

Isotonic regression produced 122 unique probabilities for month 6. Because several applications can now receive identical probabilities, ties may affect which cases are selected for investigation.
We should not choose our calibration method based on Brier score alone. Fraud investigation capacity matters just as much.

### Evaluate Calibration at Fixed Investigation Budgets

In [ ]:

# ============================================================
# Milestone 07 — Cell 8
# Operational Impact of HGB Probability Calibration
# ============================================================

import numpy as np
import pandas as pd

# All predictions refer to evaluation month 6.
y = np.asarray(y_evaluation, dtype=np.int8)

methods = {
    "Uncalibrated HGB": hgb_evaluation_raw,
    "Sigmoid HGB": hgb_evaluation_platt,
    "Isotonic HGB": hgb_evaluation_isotonic,
}

budgets = [0.01, 0.05]

n = len(y)
total_fraud = int(y.sum())

assert n == 108168
assert total_fraud == 1450

results = []

for method_name, predictions in methods.items():

    scores = np.asarray(predictions, dtype=np.float64)

    assert len(scores) == n
    assert np.isfinite(scores).all()

    # Stable ordering provides deterministic tie handling.
    ranked_indices = np.argsort(
        -scores,
        kind="stable",
    )

    for budget in budgets:

        k = int(np.floor(n * budget))

        selected_indices = ranked_indices[:k]

        fraud_caught = int(
            y[selected_indices].sum()
        )

        precision_at_k = fraud_caught / k
        recall_at_k = fraud_caught / total_fraud

        # Identify ties at the investigation cutoff.
        cutoff_score = scores[ranked_indices[k - 1]]

        above_cutoff = scores > cutoff_score
        tied_at_cutoff = scores == cutoff_score

        count_above = int(above_cutoff.sum())
        count_tied = int(tied_at_cutoff.sum())

        tied_slots = k - count_above

        tie_crosses_cutoff = tied_slots < count_tied

        # Possible detection range when the cutoff
        # splits applications with identical scores.
        fraud_above = int(y[above_cutoff].sum())
        fraud_tied = int(y[tied_at_cutoff].sum())
        nonfraud_tied = count_tied - fraud_tied

        minimum_caught = fraud_above + max(
            0,
            tied_slots - nonfraud_tied,
        )

        maximum_caught = fraud_above + min(
            tied_slots,
            fraud_tied,
        )

        results.append({
            "Method": method_name,
            "Budget": f"{budget:.0%}",
            "Investigations": k,
            "Fraud Caught": fraud_caught,
            "Precision": precision_at_k,
            "Recall": recall_at_k,
            "Cutoff Tie": tie_crosses_cutoff,
            "Min Caught": minimum_caught,
            "Max Caught": maximum_caught,
        })

operational_results = pd.DataFrame(results)

print("MILESTONE 07 — HGB OPERATIONAL CALIBRATION")
print("=" * 70)

display(
    operational_results.round(4)
)

print("\nINTERPRETATION")
print("-" * 70)

for budget in ["1%", "5%"]:

    subset = operational_results[
        operational_results["Budget"] == budget
    ]

    print(f"\nInvestigation budget: {budget}")

    for _, row in subset.iterrows():

        print(
            f"  {row['Method']}: "
            f"{row['Fraud Caught']} fraud detected "
            f"from {row['Investigations']} investigations"
        )

        if row["Cutoff Tie"]:
            print(
                "    WARNING: Ties cross the cutoff. "
                f"Possible detections: "
                f"{row['Min Caught']}–{row['Max Caught']}"
            )

print("\nEvaluation month: 6")
print("Evaluation status: EXPLORATORY")
print("Month 7 accessed: NO")

print("\nStep 07.32: PASS")


## Calibration Summary
What we've learned

- **Sigmoid calibration preserves ranking.** It produces exactly the same fraud detections as uncalibrated HGB at both investigation budgets.
- **Isotonic calibration improves Brier score and log loss,** but its 1% result is lower.
- **Isotonic's apparent three-case advantage at 5% is not reliable.** Its tied predictions cross the investigation cutoff, meaning results depend on how tied applications are selected.

At the 5% budget, isotonic calibration produced 685 detections with stable row-order tie-breaking, but other tie-breaking choices could yield between 675 and 739 detections on this dataset.

**Provisional HGB finding:** Sigmoid is attractive when preserving the existing fraud ranking matters. Isotonic is promising for probability accuracy, but requires an explicit tie-breaking policy before operational use.

### Train the Weighted PyTorch MLP

In [ ]:

# ============================================================
# Milestone 07 — Cell 9
# Train Full Class-Weighted PyTorch MLP
# Training data: Months 0-4 ONLY
# ============================================================

import time

from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader

from src.models.mlp import FraudMLP
from src.data.dataset import arrays_to_tensor_dataset
from src.training.loops import train_one_epoch

# 1. Verify the training dataset.
assert X_train.shape == (675666, 55)
assert y_train.shape == (675666,)
assert int(y_train.sum()) == 6740

assert X_train.dtype == np.float32
assert y_train.dtype == np.float32

# 2. Set reproducibility controls.
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")

# 3. Calculate class weighting from training months only.
positive_count = int(y_train.sum())
negative_count = len(y_train) - positive_count

POS_WEIGHT = negative_count / positive_count

assert positive_count == 6740
assert negative_count == 668926

print("MILESTONE 07 — WEIGHTED MLP TRAINING")
print("=" * 60)

print("Training months: 0-4")
print("Training applications:", f"{len(y_train):,}")
print("Fraud cases:", f"{positive_count:,}")
print("Non-fraud cases:", f"{negative_count:,}")
print("Positive-class weight:", f"{POS_WEIGHT:.6f}")

# 4. Construct the training dataset.
train_dataset = arrays_to_tensor_dataset(
    X_train,
    y_train,
)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
)

# 5. Initialize a NEW neural network.
# The original Milestone 06 checkpoint remains untouched.
mlp_dev = FraudMLP(
    input_dim=X_train.shape[1],
    dropout_rate=0.10,
).to(DEVICE)

parameter_count = sum(
    p.numel()
    for p in mlp_dev.parameters()
    if p.requires_grad
)

assert parameter_count == 5697

# 6. Configure weighted binary cross-entropy.
positive_weight_tensor = torch.tensor(
    POS_WEIGHT,
    dtype=torch.float32,
    device=DEVICE,
)

mlp_loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=positive_weight_tensor,
)

mlp_optimizer = Adam(
    mlp_dev.parameters(),
    lr=0.001,
)

# 7. Train for five epochs.
EPOCHS = 5

mlp_training_history = []

start_time = time.perf_counter()

for epoch in range(1, EPOCHS + 1):

    epoch_start = time.perf_counter()

    training_loss = train_one_epoch(
        model=mlp_dev,
        dataloader=train_loader,
        optimizer=mlp_optimizer,
        loss_fn=mlp_loss_fn,
        device=DEVICE,
    )

    epoch_seconds = (
        time.perf_counter() - epoch_start
    )

    mlp_training_history.append({
        "epoch": epoch,
        "training_loss": training_loss,
        "seconds": epoch_seconds,
    })

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Weighted Loss: {training_loss:.6f} | "
        f"Time: {epoch_seconds:.2f}s",
        flush=True,
    )

mlp_training_seconds = (
    time.perf_counter() - start_time
)

# 8. Verify successful training.
assert len(mlp_training_history) == 5

assert all(
    np.isfinite(row["training_loss"])
    for row in mlp_training_history
)

# 9. Switch to inference mode.
mlp_dev.eval()

print("\nTRAINING VERIFICATION")
print("-" * 60)

print("Architecture: 55 -> 64 -> 32 -> 1")
print("Trainable parameters:", parameter_count)
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Epochs completed:", len(mlp_training_history))
print("Total training time:", f"{mlp_training_seconds:.2f}s")

print("\nPreprocessing fit: Months 0-4 ONLY")
print("MLP training: Months 0-4 ONLY")
print("Month 5 used for training: NO")
print("Month 6 used for training: NO")
print("Calibration fitted: NO")
print("Month 7 accessed: NO")

print("\nStep 07.33: PASS")


### Save the Trained MLP Checkpoint

In [ ]:

# ============================================================
# Milestone 07 — Cell 10
# Save Verified Weighted MLP Checkpoint and Manifest
# ============================================================

import hashlib
import json
import subprocess
from pathlib import Path

# 1. Define versioned local artifact paths.
ARTIFACT_DIR = PROJECT_ROOT / "artifacts/milestone_07"

checkpoint_path = (
    ARTIFACT_DIR / "m07_weighted_mlp_m0to4_v01.pt"
)

manifest_path = (
    ARTIFACT_DIR / "m07_weighted_mlp_m0to4_v01.json"
)

# 2. Prevent accidental overwrites.
assert not checkpoint_path.exists(), (
    "Checkpoint already exists. Stop before overwriting."
)

assert not manifest_path.exists(), (
    "Manifest already exists. Stop before overwriting."
)

# 3. Confirm both files are protected by .gitignore.
for path in [checkpoint_path, manifest_path]:

    relative_path = path.relative_to(
        PROJECT_ROOT
    ).as_posix()

    result = subprocess.run(
        ["git", "check-ignore", "-q", "--", relative_path],
        cwd=PROJECT_ROOT,
    )

    assert result.returncode == 0, (
        f"Artifact is not Git-ignored: {relative_path}"
    )

# 4. Verify the trained model.
assert len(mlp_training_history) == 5
assert int(y_train.sum()) == 6740
assert X_train.shape == (675666, 55)
assert parameter_count == 5697

# 5. Save only the model weights.
# No customer records or prediction arrays are included.
ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

mlp_dev.eval()

torch.save(
    mlp_dev.state_dict(),
    checkpoint_path,
)

# 6. Calculate checkpoint integrity checksum.
digest = hashlib.sha256()

with checkpoint_path.open("rb") as file:
    for chunk in iter(
        lambda: file.read(1024 * 1024),
        b"",
    ):
        digest.update(chunk)

checkpoint_sha256 = digest.hexdigest()

# 7. Create the reproducibility manifest.
manifest = {
    "milestone": "07",
    "model": "FraudMLP",
    "purpose": "calibration_development",
    "architecture": [55, 64, 32, 1],
    "trainable_parameters": 5697,
    "training_months": [0, 1, 2, 3, 4],
    "calibration_month": 5,
    "evaluation_month": 6,
    "holdout_month": 7,
    "holdout_status": "SEALED",
    "training_records": 675666,
    "training_fraud_cases": 6740,
    "epochs": 5,
    "batch_size": 1024,
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "dropout": 0.10,
    "seed": 42,
    "positive_class_weight": float(POS_WEIGHT),
    "device": str(DEVICE),
    "pytorch_version": torch.__version__,
    "training_seconds": float(mlp_training_seconds),
    "training_history": mlp_training_history,
    "checkpoint_file": checkpoint_path.name,
    "checkpoint_sha256": checkpoint_sha256,
}

manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

# 8. Verify saved artifacts.
assert checkpoint_path.is_file()
assert manifest_path.is_file()

saved_manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

assert (
    saved_manifest["checkpoint_sha256"]
    == checkpoint_sha256
)

assert saved_manifest["holdout_status"] == "SEALED"

print("MILESTONE 07 — MLP CHECKPOINT VERIFICATION")
print("=" * 60)

print("Checkpoint:", checkpoint_path.name)
print("Manifest:", manifest_path.name)
print("Checkpoint SHA256:", checkpoint_sha256)
print("Model parameters:", parameter_count)
print("Epochs saved:", len(mlp_training_history))
print("Artifacts Git-ignored: PASS")
print("Original M06 checkpoint modified: NO")
print("Month-7 records accessed: 0")

print("\nStep 07.34: PASS")


### Generate MLP Calibration and Evaluation Predictions

In [ ]:

# ============================================================
# Milestone 07 — Cell 11
# Weighted MLP: Uncalibrated Prediction Evaluation
# ============================================================

from torch.utils.data import DataLoader

from src.data.dataset import arrays_to_tensor_dataset
from src.training.loops import validate_one_epoch

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)

# 1. Confirm model and data are ready.
assert X_calibration.shape == (119323, 55)
assert X_evaluation.shape == (108168, 55)

assert int(y_calibration.sum()) == 1411
assert int(y_evaluation.sum()) == 1450

assert len(mlp_training_history) == 5

mlp_dev.eval()

# 2. Define reusable inference function.
# No gradient updates or model retraining.
def predict_mlp_partition(X, y):

    dataset = arrays_to_tensor_dataset(X, y)

    loader = DataLoader(
        dataset,
        batch_size=4096,
        shuffle=False,
        num_workers=0,
    )

    weighted_loss, actual, probabilities = (
        validate_one_epoch(
            model=mlp_dev,
            dataloader=loader,
            loss_fn=mlp_loss_fn,
            device=DEVICE,
        )
    )

    # Check predictions remain aligned with labels.
    assert np.array_equal(actual, y)

    assert probabilities.shape == y.shape

    assert np.isfinite(probabilities).all()

    assert np.all(
        (probabilities >= 0)
        & (probabilities <= 1)
    )

    return probabilities, weighted_loss


# 3. Generate month-5 predictions.
# This is OUT-OF-SAMPLE relative to model training.
mlp_calibration_raw, mlp_calibration_loss = (
    predict_mlp_partition(
        X_calibration,
        y_calibration,
    )
)

# 4. Generate month-6 predictions.
# These will support exploratory evaluation.
mlp_evaluation_raw, mlp_evaluation_loss = (
    predict_mlp_partition(
        X_evaluation,
        y_evaluation,
    )
)

# 5. Evaluate uncalibrated predictions.
results = []

for name, labels, predictions, weighted_loss in [
    (
        "Calibration (month 5)",
        y_calibration,
        mlp_calibration_raw,
        mlp_calibration_loss,
    ),
    (
        "Evaluation (month 6)",
        y_evaluation,
        mlp_evaluation_raw,
        mlp_evaluation_loss,
    ),
]:

    results.append({
        "Dataset": name,
        "PR-AUC": average_precision_score(
            labels, predictions
        ),
        "ROC-AUC": roc_auc_score(
            labels, predictions
        ),
        "Brier": brier_score_loss(
            labels, predictions
        ),
        "Log Loss": log_loss(
            labels, predictions
        ),
        "Actual Fraud Rate": float(
            np.mean(labels)
        ),
        "Mean Predicted": float(
            np.mean(predictions)
        ),
        "Weighted BCE Loss": weighted_loss,
    })


mlp_uncalibrated_results = pd.DataFrame(results)

# 6. Display results.
print("MILESTONE 07 — UNCALIBRATED WEIGHTED MLP")
print("=" * 65)

print("Model trained on: Months 0-4")
print("Positive-class weight:", f"{POS_WEIGHT:.6f}")
print("Calibration predictions: Month 5")
print("Evaluation predictions: Month 6")

print("\nUNCALIBRATED MODEL PERFORMANCE")
print("-" * 65)

display(
    mlp_uncalibrated_results.round(6)
)

print("\nPrediction alignment: PASS")
print("Model weights updated: NO")
print("Calibration fitted: NO")
print("Month-6 evaluation status: EXPLORATORY")
print("Month-7 records accessed: 0")

print("\nStep 07.35: PASS")


### Restart Recovery Bundle

In [ ]:

# ============================================================
# Milestone 07 — Cell 12
# Save Restart Recovery Bundle
# ============================================================

import hashlib
import json
import subprocess
import sys
from pathlib import Path

import joblib
import numpy as np

# 1. Verify environment and project.
assert Path(sys.prefix).resolve() == (
    PROJECT_ROOT / ".venv"
).resolve(), "Wrong Python environment"

assert len(mlp_training_history) == 5
assert hgb_dev.n_iter_ == 100

assert len(y_calibration) == 119323
assert len(y_evaluation) == 108168

assert int(y_calibration.sum()) == 1411
assert int(y_evaluation.sum()) == 1450

# 2. Define a new, versioned recovery directory.
RECOVERY_DIR = (
    PROJECT_ROOT
    / "artifacts/milestone_07/restart_bundle_v01"
)

assert not RECOVERY_DIR.exists(), (
    "Recovery bundle already exists. Do not overwrite."
)

relative_dir = RECOVERY_DIR.relative_to(
    PROJECT_ROOT
).as_posix()

assert subprocess.run(
    ["git", "check-ignore", "-q", "--",
     relative_dir + "/models.joblib"],
    cwd=PROJECT_ROOT,
).returncode == 0, "Recovery artifacts are not Git-ignored"

# 3. Verify the previously saved MLP checkpoint.
mlp_manifest_path = (
    PROJECT_ROOT
    / "artifacts/milestone_07/"
      "m07_weighted_mlp_m0to4_v01.json"
)

mlp_manifest = json.loads(
    mlp_manifest_path.read_text(encoding="utf-8")
)

mlp_checkpoint = (
    mlp_manifest_path.parent
    / mlp_manifest["checkpoint_file"]
)

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()

assert mlp_checkpoint.is_file()

assert file_sha256(mlp_checkpoint) == (
    mlp_manifest["checkpoint_sha256"]
), "MLP checkpoint checksum mismatch"

# 4. Collect fitted model and preprocessing objects.
models = {
    "preprocessor": preprocessor,
    "hgb_dev": hgb_dev,
    "hgb_platt": hgb_platt,
    "hgb_isotonic": hgb_isotonic,
}

# 5. Collect development-only predictions and labels.
arrays = {
    "hgb_m5_raw": hgb_calibration_raw,
    "hgb_m6_raw": hgb_evaluation_raw,
    "hgb_m5_sigmoid": hgb_calibration_platt,
    "hgb_m6_sigmoid": hgb_evaluation_platt,
    "hgb_m5_isotonic": hgb_calibration_isotonic,
    "hgb_m6_isotonic": hgb_evaluation_isotonic,
    "mlp_m5_raw": mlp_calibration_raw,
    "mlp_m6_raw": mlp_evaluation_raw,
    "y_m5": y_calibration,
    "y_m6": y_evaluation,
    "feature_names": np.asarray(
        feature_names, dtype=str
    ),
}

assert all(
    np.asarray(arrays[key]).shape == (119323,)
    for key in [
        "hgb_m5_raw",
        "hgb_m5_sigmoid",
        "hgb_m5_isotonic",
        "mlp_m5_raw",
        "y_m5",
    ]
)

assert all(
    np.asarray(arrays[key]).shape == (108168,)
    for key in [
        "hgb_m6_raw",
        "hgb_m6_sigmoid",
        "hgb_m6_isotonic",
        "mlp_m6_raw",
        "y_m6",
    ]
)

assert len(arrays["feature_names"]) == 55

# 6. Save recovered objects and arrays.
RECOVERY_DIR.mkdir(parents=True)

models_path = RECOVERY_DIR / "models.joblib"
predictions_path = RECOVERY_DIR / "predictions.npz"
manifest_path = RECOVERY_DIR / "manifest.json"

joblib.dump(
    models,
    models_path,
    compress=3,
)

np.savez_compressed(
    predictions_path,
    **arrays,
)

# 7. Record integrity checks and provenance.
recovery_manifest = {
    "milestone": "07",
    "bundle_version": "v01",
    "status": "pre_restart_checkpoint",
    "training_months": [0, 1, 2, 3, 4],
    "calibration_month": 5,
    "evaluation_month": 6,
    "holdout_month": 7,
    "holdout_status": "SEALED",
    "models_sha256": file_sha256(models_path),
    "predictions_sha256": file_sha256(predictions_path),
    "mlp_checkpoint_file": mlp_checkpoint.name,
    "mlp_checkpoint_sha256": file_sha256(
        mlp_checkpoint
    ),
    "hgb_training_seconds": float(
        hgb_training_seconds
    ),
    "mlp_training_seconds": float(
        mlp_training_seconds
    ),
    "positive_class_weight": float(POS_WEIGHT),
    "python_version": sys.version.split()[0],
}

manifest_path.write_text(
    json.dumps(recovery_manifest, indent=2),
    encoding="utf-8",
)

# 8. Independently verify saved files.
assert file_sha256(models_path) == (
    recovery_manifest["models_sha256"]
)

assert file_sha256(predictions_path) == (
    recovery_manifest["predictions_sha256"]
)

with np.load(
    predictions_path,
    allow_pickle=False,
) as check:
    assert set(check.files) == set(arrays)
    assert np.array_equal(
        check["y_m5"], y_calibration
    )
    assert np.array_equal(
        check["y_m6"], y_evaluation
    )

print("MILESTONE 07 — RESTART RECOVERY CHECKPOINT")
print("=" * 60)
print("Recovery bundle:", RECOVERY_DIR.name)
print("Fitted models/preprocessor: SAVED")
print("Development predictions: SAVED")
print("MLP checkpoint: VERIFIED")
print("Manifest: SAVED")
print("Checksums: VERIFIED")
print("Git privacy protection: PASS")
print("Month-7 records saved: 0")
print("Restart recovery bundle: READY")
print("\nStep 07.36: PASS")


### Restore the Recovery Bundle

In [ ]:

# ============================================================
# Milestone 07 — Cell 13
# Restore All Saved Development Models and Predictions
# NO RETRAINING
# ============================================================

import hashlib
import json
import subprocess
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import average_precision_score

# 1. Find project root and verify the Jupyter environment.
current = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        p for p in (current, *current.parents)
        if (p / "configs/milestone_07_split_config.json").is_file()
    ),
    None,
)

assert PROJECT_ROOT is not None, "Project root not found"

assert Path(sys.prefix).resolve() == (
    PROJECT_ROOT / ".venv"
).resolve(), "Wrong Jupyter kernel"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.mlp import FraudMLP

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=PROJECT_ROOT,
    text=True,
).strip()

assert branch == "milestone-07-calibration-business-costs", (
    f"Unexpected branch: {branch}"
)

# 2. Locate recovery artifacts.
ARTIFACT_DIR = PROJECT_ROOT / "artifacts/milestone_07"
RECOVERY_DIR = ARTIFACT_DIR / "restart_bundle_v01"

models_path = RECOVERY_DIR / "models.joblib"
predictions_path = RECOVERY_DIR / "predictions.npz"
manifest_path = RECOVERY_DIR / "manifest.json"

mlp_manifest_path = (
    ARTIFACT_DIR / "m07_weighted_mlp_m0to4_v01.json"
)

assert all(p.is_file() for p in [
    models_path,
    predictions_path,
    manifest_path,
    mlp_manifest_path,
]), "Recovery files missing"

# 3. Verify checksums BEFORE loading saved objects.
def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


recovery_manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

mlp_manifest = json.loads(
    mlp_manifest_path.read_text(encoding="utf-8")
)

mlp_checkpoint = (
    ARTIFACT_DIR / mlp_manifest["checkpoint_file"]
)

assert mlp_checkpoint.is_file()

assert sha256_file(models_path) == (
    recovery_manifest["models_sha256"]
)

assert sha256_file(predictions_path) == (
    recovery_manifest["predictions_sha256"]
)

assert sha256_file(mlp_checkpoint) == (
    mlp_manifest["checkpoint_sha256"]
)

assert recovery_manifest["holdout_status"] == "SEALED"

# 4. Restore the trusted, checksum-verified sklearn objects.
# joblib files must only be loaded from trusted sources.
saved_models = joblib.load(models_path)

preprocessor = saved_models["preprocessor"]
hgb_dev = saved_models["hgb_dev"]
hgb_platt = saved_models["hgb_platt"]
hgb_isotonic = saved_models["hgb_isotonic"]

assert hgb_dev.n_iter_ == 100

# 5. Restore development predictions and labels.
with np.load(
    predictions_path,
    allow_pickle=False,
) as saved:

    hgb_calibration_raw = saved["hgb_m5_raw"].copy()
    hgb_evaluation_raw = saved["hgb_m6_raw"].copy()

    hgb_calibration_platt = saved["hgb_m5_sigmoid"].copy()
    hgb_evaluation_platt = saved["hgb_m6_sigmoid"].copy()

    hgb_calibration_isotonic = saved["hgb_m5_isotonic"].copy()
    hgb_evaluation_isotonic = saved["hgb_m6_isotonic"].copy()

    mlp_calibration_raw = saved["mlp_m5_raw"].copy()
    mlp_evaluation_raw = saved["mlp_m6_raw"].copy()

    y_calibration = saved["y_m5"].copy()
    y_evaluation = saved["y_m6"].copy()

    feature_names = saved["feature_names"].copy()

# 6. Reconstruct MLP architecture and restore trained weights.
DEVICE = torch.device("cpu")

mlp_dev = FraudMLP(
    input_dim=len(feature_names),
    dropout_rate=0.10,
).to(DEVICE)

mlp_state = torch.load(
    mlp_checkpoint,
    map_location=DEVICE,
    weights_only=True,
)

mlp_dev.load_state_dict(mlp_state)
mlp_dev.eval()

POS_WEIGHT = float(
    mlp_manifest["positive_class_weight"]
)

mlp_training_history = mlp_manifest["training_history"]

# 7. Independently check restored arrays and performance.
assert len(feature_names) == 55

assert len(y_calibration) == 119323
assert len(y_evaluation) == 108168

assert int(y_calibration.sum()) == 1411
assert int(y_evaluation.sum()) == 1450

assert mlp_calibration_raw.shape == y_calibration.shape
assert mlp_evaluation_raw.shape == y_evaluation.shape

assert np.isfinite(mlp_calibration_raw).all()
assert np.isfinite(mlp_evaluation_raw).all()

assert len(mlp_training_history) == 5

hgb_pr_auc = average_precision_score(
    y_evaluation,
    hgb_evaluation_raw,
)

mlp_pr_auc = average_precision_score(
    y_evaluation,
    mlp_evaluation_raw,
)

assert abs(hgb_pr_auc - 0.144497) < 0.00005
assert abs(mlp_pr_auc - 0.163322) < 0.00005

# 8. Report successful recovery.
print("MILESTONE 07 — RESTART RECOVERY VERIFICATION")
print("=" * 60)

print("Correct environment: PASS")
print("Correct Git branch: PASS")
print("Recovery checksums: PASS")
print("Preprocessor restored: PASS")
print("HGB model restored: PASS")
print("HGB calibrators restored: PASS")
print("Weighted MLP checkpoint restored: PASS")
print("Month-5 predictions restored: PASS")
print("Month-6 predictions restored: PASS")

print("\nVERIFIED PERFORMANCE")
print(f"HGB month-6 PR-AUC: {hgb_pr_auc:.6f}")
print(f"MLP month-6 PR-AUC: {mlp_pr_auc:.6f}")

print("\nModel retraining performed: NO")
print("Raw dataset accessed: NO")
print("Month-7 holdout: SEALED")

print("\nStep 07.37: PASS")


### Weighted MLP Sigmoid Calibration

In [ ]:

# ============================================================
# Milestone 07 — Cell 14
# Weighted MLP Sigmoid (Platt) Calibration
# NO MODEL RETRAINING
# ============================================================

import numpy as np
import pandas as pd

from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
)

# ------------------------------------------------------------
# 1. Verify recovered predictions and labels
# ------------------------------------------------------------

assert mlp_calibration_raw.shape == (119323,)
assert mlp_evaluation_raw.shape == (108168,)

assert y_calibration.shape == (119323,)
assert y_evaluation.shape == (108168,)

assert int(y_calibration.sum()) == 1411
assert int(y_evaluation.sum()) == 1450

for predictions in [
    mlp_calibration_raw,
    mlp_evaluation_raw,
]:
    assert np.isfinite(predictions).all()

    assert np.all(
        (predictions >= 0)
        & (predictions <= 1)
    )

# ------------------------------------------------------------
# 2. Convert probabilities into finite log-odds
# ------------------------------------------------------------

EPSILON = 1e-7

def mlp_probability_to_logit(probabilities):

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    clipped = np.clip(
        probabilities,
        EPSILON,
        1.0 - EPSILON,
    )

    return logit(clipped).reshape(-1, 1)


X_mlp_platt_calibration = mlp_probability_to_logit(
    mlp_calibration_raw
)

# ------------------------------------------------------------
# 3. Fit sigmoid calibration on MONTH 5 ONLY
# ------------------------------------------------------------

mlp_platt = LogisticRegression(
    C=1e6,
    solver="lbfgs",
    max_iter=1000,
    random_state=42,
)

mlp_platt.fit(
    X_mlp_platt_calibration,
    y_calibration.astype(np.int8),
)

assert mlp_platt.coef_[0, 0] > 0, (
    "Unexpected negative calibration slope"
)

# ------------------------------------------------------------
# 4. Generate calibrated month-5 predictions
# IN-SAMPLE calibration diagnostics
# ------------------------------------------------------------

mlp_calibration_platt = mlp_platt.predict_proba(
    X_mlp_platt_calibration
)[:, 1]

# ------------------------------------------------------------
# 5. Apply calibrator to MONTH 6
# WITHOUT REFITTING
# ------------------------------------------------------------

X_mlp_platt_evaluation = mlp_probability_to_logit(
    mlp_evaluation_raw
)

mlp_evaluation_platt = mlp_platt.predict_proba(
    X_mlp_platt_evaluation
)[:, 1]

# ------------------------------------------------------------
# 6. Validate calibrated predictions
# ------------------------------------------------------------

for name, predictions, labels in [
    ("Month 5", mlp_calibration_platt, y_calibration),
    ("Month 6", mlp_evaluation_platt, y_evaluation),
]:
    assert predictions.shape == labels.shape, (
        f"{name}: shape mismatch"
    )

    assert np.isfinite(predictions).all(), (
        f"{name}: non-finite predictions"
    )

    assert np.all(
        (predictions >= 0)
        & (predictions <= 1)
    ), f"{name}: invalid calibrated probabilities"

# ------------------------------------------------------------
# 7. Compare uncalibrated vs sigmoid on MONTH 6
# ------------------------------------------------------------

comparison_rows = []

for name, predictions in [
    ("Uncalibrated Weighted MLP", mlp_evaluation_raw),
    ("Sigmoid Weighted MLP", mlp_evaluation_platt),
]:

    comparison_rows.append({
        "Method": name,
        "PR-AUC": average_precision_score(
            y_evaluation,
            predictions,
        ),
        "Brier": brier_score_loss(
            y_evaluation,
            predictions,
        ),
        "Log Loss": log_loss(
            y_evaluation,
            predictions,
            labels=[0, 1],
        ),
        "Mean Predicted": float(
            np.mean(predictions)
        ),
    })

mlp_sigmoid_results = pd.DataFrame(
    comparison_rows
)

# ------------------------------------------------------------
# 8. Display results
# ------------------------------------------------------------

print("MILESTONE 07 — WEIGHTED MLP SIGMOID CALIBRATION")
print("=" * 65)

print("Base model trained: Months 0-4")
print("Calibrator fitted: Month 5")
print("Exploratory evaluation: Month 6")
print("Model retraining: NONE")

print(
    "\nCalibration slope:",
    f"{mlp_platt.coef_[0, 0]:.6f}",
)

print(
    "Calibration intercept:",
    f"{mlp_platt.intercept_[0]:.6f}",
)

print("\nMONTH 6 — CALIBRATION COMPARISON")
print("-" * 65)

display(
    mlp_sigmoid_results.round(6)
)

actual_fraud_rate = float(
    np.mean(y_evaluation)
)

print(
    "\nActual fraud rate:",
    f"{actual_fraud_rate:.6f}",
)

before = comparison_rows[0]
after = comparison_rows[1]

print(
    "Brier improvement (positive = better):",
    f"{before['Brier'] - after['Brier']:.8f}",
)

print(
    "Log-loss improvement (positive = better):",
    f"{before['Log Loss'] - after['Log Loss']:.8f}",
)

print(
    "PR-AUC difference:",
    f"{after['PR-AUC'] - before['PR-AUC']:.8f}",
)

print("\nMonth 5 used for calibration: YES")
print("Month 6 used to fit calibrator: NO")
print("Evaluation status: EXPLORATORY")
print("Month 7 accessed: NO")

print("\nStep 07.38: PASS")


### Weighted MLP Isotonic Calibration

In [ ]:

# ============================================================
# Milestone 07 — Cell 15
# Weighted MLP Isotonic Calibration
# NO MODEL RETRAINING
# ============================================================

import numpy as np
import pandas as pd

from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
)

# 1. Verify the restored prediction arrays.
assert mlp_calibration_raw.shape == (119323,)
assert mlp_evaluation_raw.shape == (108168,)
assert mlp_evaluation_platt.shape == (108168,)

assert int(y_calibration.sum()) == 1411
assert int(y_evaluation.sum()) == 1450

# 2. Initialize a new isotonic calibrator.
mlp_isotonic = IsotonicRegression(
    y_min=0.0,
    y_max=1.0,
    increasing=True,
    out_of_bounds="clip",
)

# 3. Fit ONLY on month-5 predictions and labels.
# The MLP itself was trained on months 0-4.
mlp_isotonic.fit(
    mlp_calibration_raw,
    y_calibration.astype(np.int8),
)

# 4. Generate month-5 fitted predictions.
# These are IN-SAMPLE calibration diagnostics.
mlp_calibration_isotonic = mlp_isotonic.predict(
    mlp_calibration_raw
)

# 5. Apply the calibrator to month 6.
# DO NOT refit using month-6 labels.
mlp_evaluation_isotonic = mlp_isotonic.predict(
    mlp_evaluation_raw
)

# 6. Verify prediction integrity.
for name, predictions, labels in [
    ("Month 5", mlp_calibration_isotonic, y_calibration),
    ("Month 6", mlp_evaluation_isotonic, y_evaluation),
]:
    assert predictions.shape == labels.shape, (
        f"{name}: prediction shape mismatch"
    )

    assert np.isfinite(predictions).all(), (
        f"{name}: non-finite predictions"
    )

    assert np.all(
        (predictions >= 0) & (predictions <= 1)
    ), f"{name}: invalid probability range"

# 7. Compare all three methods on month 6.
methods = {
    "Uncalibrated Weighted MLP": mlp_evaluation_raw,
    "Sigmoid Weighted MLP": mlp_evaluation_platt,
    "Isotonic Weighted MLP": mlp_evaluation_isotonic,
}

actual_fraud_rate = float(np.mean(y_evaluation))

comparison_rows = []

for name, predictions in methods.items():

    comparison_rows.append({
        "Method": name,
        "PR-AUC": average_precision_score(
            y_evaluation,
            predictions,
        ),
        "Brier": brier_score_loss(
            y_evaluation,
            predictions,
        ),
        "Log Loss": log_loss(
            y_evaluation,
            predictions,
            labels=[0, 1],
        ),
        "Mean Predicted": float(
            np.mean(predictions)
        ),
        "Mean Rate Error": abs(
            float(np.mean(predictions))
            - actual_fraud_rate
        ),
    })

mlp_calibration_results = pd.DataFrame(
    comparison_rows
)

# 8. Determine exploratory metric leaders.
best_brier = mlp_calibration_results.loc[
    mlp_calibration_results["Brier"].idxmin(),
    "Method",
]

best_log_loss = mlp_calibration_results.loc[
    mlp_calibration_results["Log Loss"].idxmin(),
    "Method",
]

best_pr_auc = mlp_calibration_results.loc[
    mlp_calibration_results["PR-AUC"].idxmax(),
    "Method",
]

# 9. Display comparison.
print("MILESTONE 07 — WEIGHTED MLP CALIBRATION COMPARISON")
print("=" * 65)

print("Base model training: Months 0-4")
print("Calibrators fitted: Month 5")
print("Exploratory evaluation: Month 6")
print("Model retraining: NONE")

print("\nMONTH 6 — THREE-METHOD COMPARISON")
print("-" * 65)

display(
    mlp_calibration_results.round(6)
)

print(
    "\nActual fraud rate:",
    f"{actual_fraud_rate:.6f}",
)

print("\nLowest Brier score:", best_brier)
print("Lowest log loss:", best_log_loss)
print("Highest PR-AUC:", best_pr_auc)

# 10. Inspect isotonic score ties.
print(
    "\nIsotonic fitted thresholds:",
    len(mlp_isotonic.X_thresholds_),
)

print(
    "Unique month-6 isotonic probabilities:",
    len(np.unique(mlp_evaluation_isotonic)),
)

print("\nCalibration data: Month 5 ONLY")
print("Month 6 used for fitting: NO")
print("Evaluation status: EXPLORATORY")
print("Month 7 accessed: NO")

print("\nStep 07.39: PASS")


## Weighted MLP Calibration Summary
We've completed the probability-calibration experiments for both models. The results suggest that **sigmoid calibration is the strongest option for our weighted PyTorch MLP.**

The actual fraud rate in month 6 was **1.3405%.**

**Three key findings**

- **Sigmoid calibration reduced Brier score by approximately 87.6%.** This is a substantial improvement in probability accuracy.
- **Sigmoid preserved PR-AUC,** while isotonic calibration reduced PR-AUC because its non-strictly increasing mapping creates tied scores.
- **Isotonic produced a slightly closer average fraud rate,** but sigmoid performed better on both Brier score and log loss.

These are exploratory month-6 results, not yet a final calibration-method selection.

### Compare All Six Models at Operational Budgets

In [ ]:

# ============================================================
# Milestone 07 — Cell 16
# Six-Model Operational Comparison
# NO RETRAINING OR REFITTING
# ============================================================

import numpy as np
import pandas as pd

# 1. Collect all six month-6 prediction sets.
models = {
    "HGB - Uncalibrated": hgb_evaluation_raw,
    "HGB - Sigmoid": hgb_evaluation_platt,
    "HGB - Isotonic": hgb_evaluation_isotonic,
    "MLP - Uncalibrated": mlp_evaluation_raw,
    "MLP - Sigmoid": mlp_evaluation_platt,
    "MLP - Isotonic": mlp_evaluation_isotonic,
}

# 2. Verify development-only evaluation labels.
y = np.asarray(y_evaluation, dtype=np.int8)

assert y.shape == (108168,)
assert int(y.sum()) == 1450
assert set(np.unique(y)) == {0, 1}

n = len(y)
total_fraud = int(y.sum())

# 3. Evaluate fixed investigation budgets.
BUDGETS = [0.01, 0.05]

rows = []

for model_name, predictions in models.items():

    scores = np.asarray(
        predictions, dtype=np.float64
    )

    assert scores.shape == y.shape
    assert np.isfinite(scores).all()
    assert np.all((scores >= 0) & (scores <= 1))

    # Stable sorting gives deterministic tie handling.
    ranked = np.argsort(
        -scores, kind="stable"
    )

    for budget in BUDGETS:

        k = int(np.floor(n * budget))

        selected = ranked[:k]

        fraud_caught = int(
            y[selected].sum()
        )

        # Detect whether equal scores cross the cutoff.
        cutoff = scores[ranked[k - 1]]

        above = scores > cutoff
        tied = scores == cutoff

        above_count = int(above.sum())
        tied_count = int(tied.sum())

        slots_in_tie = k - above_count

        tie_crosses_cutoff = (
            0 < slots_in_tie < tied_count
        )

        # Calculate possible outcomes under
        # alternative tie-breaking selections.
        fraud_above = int(y[above].sum())
        fraud_tied = int(y[tied].sum())
        nonfraud_tied = tied_count - fraud_tied

        minimum_caught = fraud_above + max(
            0,
            slots_in_tie - nonfraud_tied,
        )

        maximum_caught = fraud_above + min(
            slots_in_tie,
            fraud_tied,
        )

        assert (
            minimum_caught
            <= fraud_caught
            <= maximum_caught
        )

        rows.append({
            "Model": model_name,
            "Budget": f"{budget:.0%}",
            "Investigations": k,
            "Fraud Caught": fraud_caught,
            "Precision": fraud_caught / k,
            "Recall": fraud_caught / total_fraud,
            "Cutoff Tie": tie_crosses_cutoff,
            "Min Caught": minimum_caught,
            "Max Caught": maximum_caught,
        })

# 4. Create comparison table.
operational_comparison = pd.DataFrame(rows)

print("MILESTONE 07 — SIX-MODEL OPERATIONAL COMPARISON")
print("=" * 70)

display(
    operational_comparison.round(4)
)

# 5. Summarize both investigation budgets.
for budget in ["1%", "5%"]:

    subset = operational_comparison[
        operational_comparison["Budget"] == budget
    ]

    print(f"\nINVESTIGATION BUDGET: {budget}")
    print("-" * 60)

    for _, row in subset.iterrows():

        print(
            f"{row['Model']:<23} "
            f"Fraud caught: {row['Fraud Caught']:>4} "
            f"| Recall: {row['Recall']:.2%}"
        )

        if row["Cutoff Tie"]:
            print(
                "  TIE WARNING: "
                f"Possible detections "
                f"{row['Min Caught']}–"
                f"{row['Max Caught']}"
            )

# 6. Verify study boundaries.
print("\nVERIFICATION")
print("-" * 60)
print("Models compared:", len(models))
print("Evaluation records:", n)
print("Actual fraud cases:", total_fraud)
print("Calibration methods refitted: NO")
print("Models retrained: NO")
print("Evaluation status: EXPLORATORY")
print("Month 7 accessed: NO")

print("\nStep 07.40: PASS")


## Six-Model Operational Comparison Summary
Our results confirm that the sigmoid-calibrated PyTorch MLP is the leading candidate for Milestone 07. It detects more fraud than sigmoid-calibrated HGB at both investigation budgets, while maintaining better probability calibration.

Both sigmoid-calibrated models preserve their original ranking and avoid cutoff ties.
The isotonic models introduce ties at the operational cutoffs. That creates uncertainty about which applications would be investigated, making them less straightforward to deploy under fixed investigation budgets.

**Important:** These results remain exploratory because month 6 has already been used during model development. We will not treat them as final holdout performance.

### Business-Cost Scenario Analysis
Now we move from model performance to business impact.
A fraud detection system should answer more than "How many fraudulent applications did we identify?" It should also answer:

**"Is the financial benefit of investigating these applications greater than the operational cost?"**

We'll evaluate three illustrative fraud-loss scenarios. These are **assumptions for learning and sensitivity testing,** not actual financial data from the bank fraud dataset.

In [ ]:

# ============================================================
# Milestone 07 — Cell 17
# Business-Cost Scenario Analysis
# NO RETRAINING — MONTH 6 EXPLORATORY ONLY
# ============================================================

import numpy as np
import pandas as pd

# 1. Define EXPLICITLY HYPOTHETICAL business assumptions.
INVESTIGATION_COST = 25.00
FALSE_POSITIVE_COST = 10.00
PREVENTION_RATE = 0.70

FRAUD_LOSS_SCENARIOS = {
    "Low": 250.00,
    "Base": 1000.00,
    "High": 5000.00,
}

# 2. Verify month-6 evaluation boundaries.
TOTAL_APPLICATIONS = len(y_evaluation)
TOTAL_FRAUD = int(np.sum(y_evaluation))

assert TOTAL_APPLICATIONS == 108168
assert TOTAL_FRAUD == 1450
assert 0 <= PREVENTION_RATE <= 1

# 3. Use only sigmoid-calibrated model results.
CANDIDATE_MODELS = [
    "HGB - Sigmoid",
    "MLP - Sigmoid",
]

candidate_results = operational_comparison[
    operational_comparison["Model"].isin(
        CANDIDATE_MODELS
    )
].copy()

assert len(candidate_results) == 4
assert not candidate_results["Cutoff Tie"].any()

# 4. Define expected operational cost.
def calculate_business_cost(
    investigations,
    fraud_caught,
    fraud_loss,
):

    false_positives = investigations - fraud_caught

    investigation_spending = (
        investigations * INVESTIGATION_COST
    )

    false_positive_spending = (
        false_positives * FALSE_POSITIVE_COST
    )

    # Assumption: investigating a true fraud case
    # prevents PREVENTION_RATE of its expected loss.
    residual_fraud_loss = (
        TOTAL_FRAUD - PREVENTION_RATE * fraud_caught
    ) * fraud_loss

    total_cost = (
        investigation_spending
        + false_positive_spending
        + residual_fraud_loss
    )

    return {
        "False Positives": false_positives,
        "Investigation Cost": investigation_spending,
        "False Positive Cost": false_positive_spending,
        "Residual Fraud Loss": residual_fraud_loss,
        "Total Cost": total_cost,
    }

# 5. Evaluate all three scenarios.
business_rows = []

for scenario, fraud_loss in FRAUD_LOSS_SCENARIOS.items():

    # Baseline: no applications investigated.
    no_review_cost = TOTAL_FRAUD * fraud_loss

    business_rows.append({
        "Scenario": scenario,
        "Model": "No Review",
        "Budget": "0%",
        "Investigations": 0,
        "Fraud Caught": 0,
        "False Positives": 0,
        "Total Cost": no_review_cost,
        "Savings vs No Review": 0.0,
    })

    # Compare HGB and MLP at 1% and 5% capacity.
    for _, row in candidate_results.iterrows():

        investigations = int(row["Investigations"])
        fraud_caught = int(row["Fraud Caught"])

        costs = calculate_business_cost(
            investigations=investigations,
            fraud_caught=fraud_caught,
            fraud_loss=fraud_loss,
        )

        business_rows.append({
            "Scenario": scenario,
            "Model": row["Model"],
            "Budget": row["Budget"],
            "Investigations": investigations,
            "Fraud Caught": fraud_caught,
            "False Positives": costs["False Positives"],
            "Total Cost": costs["Total Cost"],
            "Savings vs No Review": (
                no_review_cost - costs["Total Cost"]
            ),
        })

business_cost_results = pd.DataFrame(business_rows)

# 6. Verify calculated results.
assert len(business_cost_results) == 15
assert business_cost_results["Total Cost"].ge(0).all()

# 7. Display complete scenario comparison.
print("MILESTONE 07 — BUSINESS-COST SCENARIO ANALYSIS")
print("=" * 70)

print("ASSUMPTIONS — HYPOTHETICAL, NOT BANK DATA")
print("-" * 70)

print(f"Investigation cost: ${INVESTIGATION_COST:,.2f}")
print(f"False-positive cost: ${FALSE_POSITIVE_COST:,.2f}")
print(f"Fraud loss prevented: {PREVENTION_RATE:.0%}")
print(f"Evaluation applications: {TOTAL_APPLICATIONS:,}")
print(f"Confirmed fraud cases: {TOTAL_FRAUD:,}")

for scenario in FRAUD_LOSS_SCENARIOS:

    print(f"\n{scenario.upper()} FRAUD-LOSS SCENARIO")
    print("-" * 70)

    subset = business_cost_results[
        business_cost_results["Scenario"] == scenario
    ]

    display(subset.drop(columns="Scenario").round(2))

# 8. Direct MLP vs HGB comparison in base scenario.
base = business_cost_results[
    business_cost_results["Scenario"] == "Base"
]

print("\nBASE SCENARIO — MLP VS HGB")
print("-" * 70)

for budget in ["1%", "5%"]:

    hgb_cost = float(
        base.loc[
            (base["Model"] == "HGB - Sigmoid")
            & (base["Budget"] == budget),
            "Total Cost",
        ].iloc[0]
    )

    mlp_cost = float(
        base.loc[
            (base["Model"] == "MLP - Sigmoid")
            & (base["Budget"] == budget),
            "Total Cost",
        ].iloc[0]
    )

    advantage = hgb_cost - mlp_cost

    print(
        f"Budget {budget}: "
        f"MLP estimated cost advantage = "
        f"${advantage:,.2f}"
    )

# 9. Document safeguards.
print("\nVERIFICATION")
print("-" * 70)
print("Business assumptions: HYPOTHETICAL")
print("Models retrained: NO")
print("Calibrators refitted: NO")
print("Month-6 evaluation: EXPLORATORY")
print("Month-7 holdout: SEALED")

print("\nStep 07.41: PASS")


## Business-Cost Scenario Analysis Summary
Your model comparisons revealed an important relationship between investigation budgets, losses, and costs.

The findings suggest:

- **Low-loss scenario ($250):** Investigating 1% of applications is more economical than investigating 5%, under our assumptions.
- **Base scenario ($1,000):** Investigating 5% produces greater estimated savings than investigating 1%.
- **High-loss scenario ($5,000):** Investigating 5% becomes considerably more valuable.

This illustrates why **the model with the highest fraud recall is not automatically the most profitable operating policy.**

### Break-Even & Sensitivity Analysis

In [ ]:

# ============================================================
# Milestone 07 — Cell 18
# Break-Even and Business-Cost Sensitivity Analysis
# Month 6: EXPLORATORY ONLY
# ============================================================

import numpy as np
import pandas as pd

# 1. Verify existing experiment results.
assert len(y_evaluation) == 108168
assert int(y_evaluation.sum()) == 1450

assert len(operational_comparison) == 12

# All financial values remain HYPOTHETICAL.
INVESTIGATION_COST = 25.0
FALSE_POSITIVE_COST = 10.0
PREVENTION_RATE = 0.70
BASE_FRAUD_LOSS = 1000.0

CANDIDATES = ["HGB - Sigmoid", "MLP - Sigmoid"]

ops = operational_comparison[
    operational_comparison["Model"].isin(CANDIDATES)
].copy()

assert len(ops) == 4
assert not ops["Cutoff Tie"].any()

# 2. Define a reusable economic model.
def policy_cost(
    investigations,
    fraud_caught,
    fraud_loss,
    investigation_cost=25.0,
    false_positive_cost=10.0,
    prevention_rate=0.70,
):
    false_positives = investigations - fraud_caught

    operational_cost = (
        investigations * investigation_cost
        + false_positives * false_positive_cost
    )

    residual_loss = (
        int(y_evaluation.sum())
        - prevention_rate * fraud_caught
    ) * fraud_loss

    return operational_cost + residual_loss


# 3. Calculate fraud-loss break-even thresholds.
break_even_rows = []

for model in CANDIDATES:

    subset = ops[ops["Model"] == model]

    one = subset[subset["Budget"] == "1%"].iloc[0]
    five = subset[subset["Budget"] == "5%"].iloc[0]

    for row in [one, five]:

        investigations = int(row["Investigations"])
        caught = int(row["Fraud Caught"])
        false_positives = investigations - caught

        operating_cost = (
            investigations * INVESTIGATION_COST
            + false_positives * FALSE_POSITIVE_COST
        )

        break_even_loss = (
            operating_cost
            / (PREVENTION_RATE * caught)
        )

        break_even_rows.append({
            "Model": model,
            "Budget": row["Budget"],
            "Break-Even vs No Review ($)": break_even_loss,
        })

    # Incremental break-even: 5% versus 1%.
    extra_investigations = (
        int(five["Investigations"])
        - int(one["Investigations"])
    )

    extra_fraud = (
        int(five["Fraud Caught"])
        - int(one["Fraud Caught"])
    )

    extra_false_positives = (
        extra_investigations - extra_fraud
    )

    incremental_operating_cost = (
        extra_investigations * INVESTIGATION_COST
        + extra_false_positives * FALSE_POSITIVE_COST
    )

    incremental_break_even = (
        incremental_operating_cost
        / (PREVENTION_RATE * extra_fraud)
    )

    break_even_rows.append({
        "Model": model,
        "Budget": "5% vs 1%",
        "Break-Even vs No Review ($)": np.nan,
        "Incremental Break-Even ($)": incremental_break_even,
    })

break_even_results = pd.DataFrame(break_even_rows)

print("MILESTONE 07 — BREAK-EVEN ANALYSIS")
print("=" * 70)

display(break_even_results.round(2))

# 4. Sensitivity analysis for the leading MLP candidate.
# Test different investigation costs and prevention rates
# while holding fraud loss at $1,000 per case.

mlp_ops = ops[ops["Model"] == "MLP - Sigmoid"]

one = mlp_ops[mlp_ops["Budget"] == "1%"].iloc[0]
five = mlp_ops[mlp_ops["Budget"] == "5%"].iloc[0]

assert int(one["Fraud Caught"]) == 285
assert int(five["Fraud Caught"]) == 705

investigation_costs = [10.0, 25.0, 50.0]
prevention_rates = [0.30, 0.70, 0.90]

sensitivity_rows = []

for prevention in prevention_rates:
    for investigation_cost in investigation_costs:

        no_review = (
            int(y_evaluation.sum()) * BASE_FRAUD_LOSS
        )

        cost_1 = policy_cost(
            investigations=int(one["Investigations"]),
            fraud_caught=int(one["Fraud Caught"]),
            fraud_loss=BASE_FRAUD_LOSS,
            investigation_cost=investigation_cost,
            false_positive_cost=FALSE_POSITIVE_COST,
            prevention_rate=prevention,
        )

        cost_5 = policy_cost(
            investigations=int(five["Investigations"]),
            fraud_caught=int(five["Fraud Caught"]),
            fraud_loss=BASE_FRAUD_LOSS,
            investigation_cost=investigation_cost,
            false_positive_cost=FALSE_POSITIVE_COST,
            prevention_rate=prevention,
        )

        policies = {
            "No Review": no_review,
            "1%": cost_1,
            "5%": cost_5,
        }

        best_policy = min(policies, key=policies.get)

        sensitivity_rows.append({
            "Prevention Rate": f"{prevention:.0%}",
            "Investigation Cost": investigation_cost,
            "No Review Cost": no_review,
            "1% Cost": cost_1,
            "5% Cost": cost_5,
            "Lowest-Cost Policy": best_policy,
        })

sensitivity_results = pd.DataFrame(sensitivity_rows)

# 5. Display decision sensitivity matrix.
decision_matrix = sensitivity_results.pivot(
    index="Prevention Rate",
    columns="Investigation Cost",
    values="Lowest-Cost Policy",
)

print("\nMLP — BUSINESS DECISION SENSITIVITY")
print("=" * 70)

print("Assumed fraud loss per case: $1,000")
print("Additional false-positive cost: $10")
print("Each cell shows the lowest-cost policy.")

display(decision_matrix)

print("\nFULL SENSITIVITY RESULTS")
display(sensitivity_results.round(2))

# 6. Verify consistency with our previous cost analysis.
base_row = sensitivity_results[
    (sensitivity_results["Prevention Rate"] == "70%")
    & (sensitivity_results["Investigation Cost"] == 25.0)
].iloc[0]

assert np.isclose(base_row["1% Cost"], 1285485.0)
assert np.isclose(base_row["5% Cost"], 1138730.0)
assert base_row["Lowest-Cost Policy"] == "5%"

# 7. Report experiment safeguards.
print("\nVERIFICATION")
print("-" * 70)

print("Break-even calculations: PASS")
print("Base scenario matches Step 07.41: PASS")
print("Sensitivity scenarios tested:", len(sensitivity_results))
print("Financial assumptions: HYPOTHETICAL")
print("Models retrained: NO")
print("Calibrators refitted: NO")
print("Month-6 evaluation: EXPLORATORY")
print("Month-7 holdout: SEALED")

print("\nStep 07.42: PASS")


### Cost-Based Investigation Threshold Optimization

In [ ]:

# ============================================================
# Milestone 07 — Cell 19
# Cost-Based Investigation Threshold Optimization
# Month 6: EXPLORATORY ONLY
# NO RETRAINING / NO LABEL-BASED THRESHOLD FITTING
# ============================================================

import numpy as np
import pandas as pd

# 1. Verify previously defined business assumptions.
assert INVESTIGATION_COST == 25.0
assert FALSE_POSITIVE_COST == 10.0
assert PREVENTION_RATE == 0.70
assert BASE_FRAUD_LOSS == 1000.0

assert len(y_evaluation) == 108168
assert int(np.sum(y_evaluation)) == 1450

# 2. Calculate the economically justified threshold.
#
# Review when:
# p * prevention_rate * fraud_loss
# > investigation_cost + (1-p) * false_positive_cost

benefit_if_fraud = (
    PREVENTION_RATE * BASE_FRAUD_LOSS
)

economic_threshold = (
    (INVESTIGATION_COST + FALSE_POSITIVE_COST)
    / (benefit_if_fraud + FALSE_POSITIVE_COST)
)

assert 0 < economic_threshold < 1

print("MILESTONE 07 — COST-BASED THRESHOLD OPTIMIZATION")
print("=" * 70)

print(
    "Economic probability threshold:",
    f"{economic_threshold:.4%}"
)

# 3. Compare the two sigmoid-calibrated models.
# These models were trained on months 0-4.
# Calibration was fitted on month 5 only.

candidate_predictions = {
    "HGB - Sigmoid": hgb_evaluation_platt,
    "MLP - Sigmoid": mlp_evaluation_platt,
}

n = len(y_evaluation)

capacity_limits = {
    "1% Capacity": int(np.floor(n * 0.01)),
    "5% Capacity": int(np.floor(n * 0.05)),
    "Uncapped": n,
}

policy_rows = []

# 4. Select investigations using predicted risk ONLY.
for model_name, probabilities in candidate_predictions.items():

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    assert probabilities.shape == (n,)
    assert np.isfinite(probabilities).all()
    assert np.all(
        (probabilities >= 0) & (probabilities <= 1)
    )

    # Predicted economic benefit of investigating
    # each application.
    expected_benefit = (
        probabilities * benefit_if_fraud
        - INVESTIGATION_COST
        - (1 - probabilities) * FALSE_POSITIVE_COST
    )

    # Select only applications with positive
    # expected economic benefit.
    eligible = np.flatnonzero(
        expected_benefit > 0
    )

    # Highest expected benefit first.
    # Stable sorting makes tie handling deterministic.
    ranking = eligible[
        np.argsort(
            -expected_benefit[eligible],
            kind="stable",
        )
    ]

    print(f"\nMODEL: {model_name}")
    print("-" * 70)
    print("Applications above threshold:", len(eligible))

    for policy_name, capacity in capacity_limits.items():

        chosen = ranking[:capacity]
        investigations = len(chosen)

        # Expected value uses predictions only.
        expected_savings = float(
            expected_benefit[chosen].sum()
        )

        # Check whether tied scores cross capacity.
        cutoff_tie = (
            investigations > 0
            and investigations < len(ranking)
            and probabilities[ranking[investigations - 1]]
                == probabilities[ranking[investigations]]
        )

        # Exploratory evaluation starts here.
        # Month-6 labels did NOT determine who
        # was selected for investigation.
        fraud_caught = int(
            np.asarray(y_evaluation)[chosen].sum()
        )

        false_positives = (
            investigations - fraud_caught
        )

        realized_savings = (
            fraud_caught * benefit_if_fraud
            - investigations * INVESTIGATION_COST
            - false_positives * FALSE_POSITIVE_COST
        )

        no_review_cost = (
            int(np.sum(y_evaluation))
            * BASE_FRAUD_LOSS
        )

        realized_total_cost = (
            no_review_cost - realized_savings
        )

        policy_rows.append({
            "Model": model_name,
            "Policy": policy_name,
            "Capacity": capacity,
            "Investigations": investigations,
            "Fraud Caught": fraud_caught,
            "False Positives": false_positives,
            "Expected Savings": expected_savings,
            "Exploratory Savings": realized_savings,
            "Exploratory Total Cost": realized_total_cost,
            "Cutoff Tie": cutoff_tie,
        })

# 5. Create policy comparison.
threshold_policy_results = pd.DataFrame(
    policy_rows
)

assert len(threshold_policy_results) == 6

assert (
    threshold_policy_results["Investigations"]
    <= threshold_policy_results["Capacity"]
).all()

assert (
    threshold_policy_results["Expected Savings"]
    >= -1e-8
).all()

print("\nCOST-BASED INVESTIGATION POLICIES")
print("=" * 70)

display(
    threshold_policy_results.round(2)
)

# 6. Compare policies within each model.
print("\nEXPLORATORY POLICY COMPARISON")
print("-" * 70)

for model_name in candidate_predictions:

    subset = threshold_policy_results[
        threshold_policy_results["Model"]
        == model_name
    ]

    for _, row in subset.iterrows():

        print(
            f"{model_name:<16} "
            f"{row['Policy']:<12} "
            f"Reviewed: {int(row['Investigations']):>6,} "
            f"| Fraud caught: {int(row['Fraud Caught']):>4} "
            f"| Savings: ${row['Exploratory Savings']:>12,.2f}"
        )

        if row["Cutoff Tie"]:
            print(
                "  WARNING: Equal probabilities cross "
                "the investigation cutoff."
            )

# 7. Final safeguards.
print("\nVERIFICATION")
print("-" * 70)

print("Threshold derived from business costs: YES")
print("Month-6 labels used to choose threshold: NO")
print("Month-6 labels used for evaluation: YES")
print("Models retrained: NO")
print("Calibrators refitted: NO")
print("Financial assumptions: HYPOTHETICAL")
print("Month-6 evaluation: EXPLORATORY")
print("Month-7 holdout: SEALED")

print("\nStep 07.43: PASS")


## Cost-Based Threshold Optimization Summary
Your results confirm that the sigmoid-calibrated MLP remains our leading candidate under the hypothetical business assumptions.

One particularly important result: the MLP's uncapped policy requires **1,503 additional investigations** compared with its 5% capacity policy, but produces only **$3,485 in additional estimated savings.**

That means we should evaluate operational capacity and workload before recommending an uncapped policy.

Also, the models' predicted savings differ from the savings calculated using actual month-6 labels. This is a reason to examine calibration specifically among high-risk applications before finalizing deployment decisions.

### Preserve Calibration and Business Results

In [ ]:

# ============================================================
# Milestone 07 — Cell 20
# Save Calibration and Business Policy Artifacts
# NO RETRAINING — MONTH 7 REMAINS SEALED
# ============================================================

import hashlib
import json
import subprocess
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

# 1. Define versioned artifact directory.
SAVE_DIR = (
    PROJECT_ROOT
    / "artifacts/milestone_07/calibration_policy_v01"
)

assert not SAVE_DIR.exists(), (
    "Version v01 already exists. Do not overwrite."
)

# 2. Verify Git privacy protection.
relative_test_path = (
    SAVE_DIR / "mlp_calibrators.joblib"
).relative_to(PROJECT_ROOT).as_posix()

assert subprocess.run(
    ["git", "check-ignore", "-q", "--",
     relative_test_path],
    cwd=PROJECT_ROOT,
).returncode == 0, "Artifacts are not Git-ignored."

# 3. Verify important experiment results.
assert len(mlp_calibration_results) == 3
assert len(operational_comparison) == 12
assert len(business_cost_results) == 15
assert len(break_even_results) == 6
assert len(sensitivity_results) == 9
assert len(threshold_policy_results) == 6

assert mlp_evaluation_platt.shape == (108168,)
assert mlp_evaluation_isotonic.shape == (108168,)

assert mlp_calibration_platt.shape == (119323,)
assert mlp_calibration_isotonic.shape == (119323,)

assert int(y_calibration.sum()) == 1411
assert int(y_evaluation.sum()) == 1450

# 4. Create directory.
SAVE_DIR.mkdir(parents=True)

# 5. Save fitted MLP calibrators.
calibrators_path = SAVE_DIR / "mlp_calibrators.joblib"

joblib.dump(
    {
        "sigmoid": mlp_platt,
        "isotonic": mlp_isotonic,
    },
    calibrators_path,
    compress=3,
)

# 6. Save development-only calibrated predictions.
predictions_path = SAVE_DIR / "mlp_calibrated_scores.npz"

np.savez_compressed(
    predictions_path,
    mlp_m5_sigmoid=mlp_calibration_platt,
    mlp_m6_sigmoid=mlp_evaluation_platt,
    mlp_m5_isotonic=mlp_calibration_isotonic,
    mlp_m6_isotonic=mlp_evaluation_isotonic,
)

# 7. Save analytical result tables.
tables = {
    "mlp_calibration_results.csv":
        mlp_calibration_results,
    "operational_comparison.csv":
        operational_comparison,
    "business_cost_results.csv":
        business_cost_results,
    "break_even_results.csv":
        break_even_results,
    "sensitivity_results.csv":
        sensitivity_results,
    "threshold_policy_results.csv":
        threshold_policy_results,
}

for filename, dataframe in tables.items():
    dataframe.to_csv(
        SAVE_DIR / filename,
        index=False,
    )

# 8. Calculate checksums.
def calculate_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()

saved_files = [
    calibrators_path,
    predictions_path,
    *[SAVE_DIR / name for name in tables],
]

checksums = {
    path.name: calculate_sha256(path)
    for path in saved_files
}

# 9. Create reproducibility manifest.
policy_manifest = {
    "milestone": "07",
    "version": "v01",
    "status": "EXPLORATORY_NOT_FINAL_POLICY",
    "base_training_months": [0, 1, 2, 3, 4],
    "calibration_month": 5,
    "evaluation_month": 6,
    "holdout_month": 7,
    "holdout_status": "SEALED",
    "candidate_models": [
        "HGB - Sigmoid",
        "MLP - Sigmoid",
    ],
    "financial_assumptions": {
        "investigation_cost": INVESTIGATION_COST,
        "false_positive_cost": FALSE_POSITIVE_COST,
        "prevention_rate": PREVENTION_RATE,
        "base_fraud_loss": BASE_FRAUD_LOSS,
        "assumptions_are_hypothetical": True,
    },
    "economic_probability_threshold": float(
        economic_threshold
    ),
    "model_selected_for_deployment": False,
    "file_checksums": checksums,
}

manifest_path = SAVE_DIR / "manifest.json"

manifest_path.write_text(
    json.dumps(policy_manifest, indent=2),
    encoding="utf-8",
)

# 10. Verify every saved file.
for path in saved_files:
    assert path.is_file()
    assert calculate_sha256(path) == checksums[path.name]

with np.load(
    predictions_path,
    allow_pickle=False,
) as saved:
    assert saved["mlp_m6_sigmoid"].shape == (108168,)
    assert saved["mlp_m5_isotonic"].shape == (119323,)

saved_manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

assert saved_manifest["holdout_status"] == "SEALED"

assert (
    saved_manifest["model_selected_for_deployment"]
    is False
)

# 11. Report results.
print("MILESTONE 07 — CALIBRATION POLICY CHECKPOINT")
print("=" * 65)

print("Artifact directory:", SAVE_DIR.name)
print("MLP calibrators: SAVED")
print("Calibrated predictions: SAVED")
print("Analysis tables:", len(tables))
print("Financial assumptions: DOCUMENTED")
print("Reproducibility manifest: SAVED")
print("Checksums: VERIFIED")
print("Git privacy: PASS")

print("\nModels retrained: NO")
print("Month 6 status: EXPLORATORY")
print("Final deployment policy selected: NO")
print("Month 7 accessed: NO")

print("\nStep 07.44: PASS")


### High-Risk Calibration Audit

In [ ]:

# ============================================================
# Milestone 07 — Cell 21
# High-Risk Probability Calibration Audit
# NO RETRAINING OR CALIBRATOR REFITTING
# Month 6: EXPLORATORY ONLY
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import brier_score_loss

# 1. Validate evaluation boundaries.
y = np.asarray(y_evaluation, dtype=np.int8)

assert y.shape == (108168,)
assert int(y.sum()) == 1450

assert np.isclose(
    economic_threshold,
    35 / 710,
    atol=1e-12,
)

models_to_audit = {
    "HGB - Sigmoid": hgb_evaluation_platt,
    "MLP - Sigmoid": mlp_evaluation_platt,
}

# 2. Define risk bands before examining outcomes.
risk_bands = [
    (0.00, 0.01, "0–1%"),
    (0.01, 0.02, "1–2%"),
    (0.02, 0.05, "2–5%"),
    (0.05, 0.10, "5–10%"),
    (0.10, 0.20, "10–20%"),
    (0.20, 0.50, "20–50%"),
    (0.50, 1.01, "50–100%"),
]

risk_rows = []
policy_rows = []

for model_name, predictions in models_to_audit.items():

    p = np.asarray(predictions, dtype=np.float64)

    assert p.shape == y.shape
    assert np.isfinite(p).all()
    assert np.all((p >= 0) & (p <= 1))

    # 3. Evaluate mutually exclusive probability bands.
    for low, high, label in risk_bands:

        mask = (p >= low) & (p < high)

        count = int(mask.sum())

        if count == 0:
            continue

        predicted_rate = float(p[mask].mean())
        observed_rate = float(y[mask].mean())

        risk_rows.append({
            "Model": model_name,
            "Risk Band": label,
            "Applications": count,
            "Expected Fraud": float(p[mask].sum()),
            "Observed Fraud": int(y[mask].sum()),
            "Predicted Rate": predicted_rate,
            "Observed Rate": observed_rate,
            "Rate Gap (pp)": (
                observed_rate - predicted_rate
            ) * 100,
        })

    # 4. Audit operational investigation pools.
    ranked = np.argsort(-p, kind="stable")

    pools = {
        "Top 1%": ranked[:1081],
        "Top 5%": ranked[:5408],
        "Economic Threshold": np.flatnonzero(
            p > economic_threshold
        ),
    }

    for pool_name, selected in pools.items():

        count = len(selected)

        expected_fraud = float(p[selected].sum())
        observed_fraud = int(y[selected].sum())

        predicted_rate = expected_fraud / count
        observed_rate = observed_fraud / count

        policy_rows.append({
            "Model": model_name,
            "Pool": pool_name,
            "Investigations": count,
            "Expected Fraud": expected_fraud,
            "Observed Fraud": observed_fraud,
            "Predicted Rate": predicted_rate,
            "Observed Rate": observed_rate,
            "Expected/Observed Ratio": (
                expected_fraud / observed_fraud
                if observed_fraud > 0
                else np.nan
            ),
            "Fraud Count Gap": (
                observed_fraud - expected_fraud
            ),
        })

# 5. Build calibration audit tables.
risk_band_audit = pd.DataFrame(risk_rows)

investigation_pool_audit = pd.DataFrame(
    policy_rows
)

assert len(investigation_pool_audit) == 6

assert (
    risk_band_audit.groupby("Model")["Applications"]
    .sum()
    .eq(108168)
    .all()
)

# 6. Display results.
print("MILESTONE 07 — HIGH-RISK CALIBRATION AUDIT")
print("=" * 72)

print("Risk bands: PREDEFINED")
print("Evaluation period: Month 6")
print("Actual fraud cases:", int(y.sum()))
print("Economic threshold:", f"{economic_threshold:.4%}")

print("\nA. CALIBRATION BY PREDICTED RISK BAND")
print("-" * 72)

display(
    risk_band_audit.round(4)
)

print("\nB. CALIBRATION WITHIN INVESTIGATION POOLS")
print("-" * 72)

display(
    investigation_pool_audit.round(4)
)

# 7. Compare global versus high-risk calibration.
print("\nC. CALIBRATION INTERPRETATION")
print("-" * 72)

for model_name in models_to_audit:

    subset = investigation_pool_audit[
        investigation_pool_audit["Model"] == model_name
    ]

    print(f"\n{model_name}")

    for _, row in subset.iterrows():

        print(
            f"{row['Pool']}: "
            f"expected {row['Expected Fraud']:.1f} fraud, "
            f"observed {int(row['Observed Fraud'])} "
            f"| gap {row['Fraud Count Gap']:+.1f}"
        )

# 8. Verification.
print("\nVERIFICATION")
print("-" * 72)

print("Models audited:", len(models_to_audit))
print("Investigation pools audited:", len(investigation_pool_audit))
print("Evaluation months: [6]")
print("Models retrained: NO")
print("Calibrators refitted: NO")
print("Business assumptions changed: NO")
print("Month 6 status: EXPLORATORY")
print("Month 7 accessed: NO")

print("\nStep 07.45: PASS")


## High Risk Calibration Audit Summary
These results provide strong additional evidence for choosing the sigmoid-calibrated PyTorch MLP as our provisional fraud-detection model.

The MLP's predicted fraud counts are considerably closer to the observed counts in the investigation pools.

However, the MLP still underestimates fraud in those pools, while HGB substantially overestimates it. Neither model is perfectly calibrated across all risk levels.

The highest risk band also has relatively few observations, so we should avoid drawing strong conclusions from that band alone.

### Freeze the Provisional Investigation Policy

In [ ]:

# ============================================================
# Milestone 07 — Cell 22
# Freeze Provisional Policy and Save Calibration Audit
# NO RETRAINING — NO HOLDOUT ACCESS
# ============================================================

import hashlib
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

# 1. Verify the exploratory results.
assert len(risk_band_audit) == 14
assert len(investigation_pool_audit) == 6
assert len(threshold_policy_results) == 6
assert len(y_evaluation) == 108168
assert int(y_evaluation.sum()) == 1450

assert INVESTIGATION_COST == 25.0
assert FALSE_POSITIVE_COST == 10.0
assert PREVENTION_RATE == 0.70
assert BASE_FRAUD_LOSS == 1000.0

assert np.isclose(
    economic_threshold,
    35.0 / 710.0,
)

# 2. Verify the selected operating policy.
candidate = threshold_policy_results[
    (threshold_policy_results["Model"] == "MLP - Sigmoid")
    & (threshold_policy_results["Policy"] == "5% Capacity")
]

assert len(candidate) == 1

candidate = candidate.iloc[0]

assert int(candidate["Investigations"]) == 5408
assert int(candidate["Fraud Caught"]) == 705
assert not bool(candidate["Cutoff Tie"])

# 3. Define paths without overwriting existing work.
audit_dir = (
    PROJECT_ROOT
    / "artifacts/milestone_07/high_risk_audit_v01"
)

policy_path = (
    PROJECT_ROOT
    / "configs/milestone_07_candidate_policy_v01.json"
)

assert not audit_dir.exists()
assert not policy_path.exists()

audit_files = {
    "risk_band_audit.csv": risk_band_audit,
    "investigation_pool_audit.csv": investigation_pool_audit,
}

# 4. Confirm private audit data remains Git-ignored.
for filename in audit_files:

    relative = (
        audit_dir / filename
    ).relative_to(PROJECT_ROOT).as_posix()

    result = subprocess.run(
        ["git", "check-ignore", "-q", "--", relative],
        cwd=PROJECT_ROOT,
    )

    assert result.returncode == 0, (
        f"Audit file not Git-ignored: {relative}"
    )

# 5. Verify previously saved model artifacts.
artifact_root = (
    PROJECT_ROOT / "artifacts/milestone_07"
)

mlp_manifest_path = (
    artifact_root / "m07_weighted_mlp_m0to4_v01.json"
)

calibration_manifest_path = (
    artifact_root / "calibration_policy_v01/manifest.json"
)

mlp_manifest = json.loads(
    mlp_manifest_path.read_text(encoding="utf-8")
)

calibration_manifest = json.loads(
    calibration_manifest_path.read_text(encoding="utf-8")
)

checkpoint_path = (
    artifact_root / mlp_manifest["checkpoint_file"]
)

calibrator_path = (
    calibration_manifest_path.parent
    / "mlp_calibrators.joblib"
)

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()

assert checkpoint_path.is_file()
assert calibrator_path.is_file()

assert sha256_file(checkpoint_path) == (
    mlp_manifest["checkpoint_sha256"]
)

assert sha256_file(calibrator_path) == (
    calibration_manifest["file_checksums"][
        "mlp_calibrators.joblib"
    ]
)

# 6. Save aggregate high-risk calibration results.
audit_dir.mkdir(parents=True)

for filename, dataframe in audit_files.items():
    dataframe.to_csv(
        audit_dir / filename,
        index=False,
    )

audit_checksums = {
    filename: sha256_file(audit_dir / filename)
    for filename in audit_files
}

# 7. Create a Git-trackable, non-sensitive policy.
policy = {
    "milestone": "07",
    "policy_version": "v01",
    "status": "FROZEN_PROVISIONAL_CANDIDATE",
    "production_approved": False,
    "model": {
        "type": "FraudMLP",
        "architecture": [55, 64, 32, 1],
        "calibration": "sigmoid_platt",
        "checkpoint_sha256": sha256_file(checkpoint_path),
        "calibrator_bundle_sha256": sha256_file(
            calibrator_path
        ),
    },
    "data_protocol": {
        "training_months": [0, 1, 2, 3, 4],
        "calibration_month": 5,
        "exploratory_evaluation_month": 6,
        "holdout_month": 7,
        "holdout_status": "SEALED",
        "month_6_used_in_candidate_comparison": True,
    },
    "business_assumptions": {
        "currency": "USD",
        "hypothetical": True,
        "investigation_cost": INVESTIGATION_COST,
        "additional_false_positive_cost": FALSE_POSITIVE_COST,
        "fraud_loss_per_case": BASE_FRAUD_LOSS,
        "prevention_rate": PREVENTION_RATE,
    },
    "decision_rule": {
        "probability_threshold": float(economic_threshold),
        "maximum_investigation_fraction": 0.05,
        "investigate_if": "calibrated_probability > threshold",
        "priority": "descending_calibrated_probability",
        "capacity_count_rule": "floor(0.05 * application_count)",
        "tie_breaker": "stable_input_order_provisional",
    },
    "exploratory_month_6": {
        "applications": 108168,
        "fraud_cases": 1450,
        "investigations": 5408,
        "fraud_caught": 705,
        "hypothetical_savings_usd": float(
            candidate["Exploratory Savings"]
        ),
    },
    "audit_file_checksums": audit_checksums,
    "limitations": [
        "Month 6 was previously used for model comparison.",
        "Financial assumptions are hypothetical.",
        "Policy is not independently holdout-validated.",
        "Production requires an immutable application-ID tie-breaker.",
        "Risk-band calibration varies across score ranges.",
    ],
}

# 8. Write and verify the frozen policy.
policy_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

policy_path.write_text(
    json.dumps(policy, indent=2),
    encoding="utf-8",
)

saved_policy = json.loads(
    policy_path.read_text(encoding="utf-8")
)

assert saved_policy == policy

assert (
    saved_policy["data_protocol"]["holdout_status"]
    == "SEALED"
)

assert (
    saved_policy["production_approved"]
    is False
)

for filename, expected_hash in audit_checksums.items():
    assert sha256_file(
        audit_dir / filename
    ) == expected_hash

# 9. Report verification.
print("MILESTONE 07 — PROVISIONAL POLICY FREEZE")
print("=" * 65)

print("Selected model: Weighted PyTorch MLP")
print("Calibration: Sigmoid / Platt")
print(
    "Economic threshold:",
    f"{economic_threshold:.4%}",
)
print("Investigation capacity: 5%")
print("Month-6 investigations:", 5408)
print("Month-6 fraud caught:", 705)

print("\nPolicy:", policy_path.name)
print("Policy status: FROZEN PROVISIONAL")
print("Calibration audit files:", len(audit_files))
print("Model checksum: VERIFIED")
print("Calibrator checksum: VERIFIED")
print("Audit checksums: VERIFIED")
print("Private audit artifacts: Git-ignored")

print("\nProduction approved: NO")
print("Independent holdout evaluation: NOT PERFORMED")
print("Month 6: EXPLORATORY")
print("Month 7: SEALED")

print("\nStep 07.46: PASS")



# Milestone 07 — Summary and Interview Preparation

## 1. Milestone Objective

The objective of Milestone 07 was to evaluate, calibrate, and establish a decision policy for a fraud-detection model while preventing data leakage and protecting an independent final holdout.

The milestone covers:

- Chronological data splitting
- Probability calibration
- Model comparison
- Fraud classification evaluation
- Decision threshold selection
- Investigation capacity constraints
- Business-cost analysis
- Reproducibility and artifact verification
- Independent final holdout evaluation

---

## 2. Dataset and Evaluation Strategy

We used the Bank Account Fraud Dataset Suite with a chronological split.

| Dataset partition | Months | Purpose |
|---|---|---|
| Training | 0–4 | Fit preprocessing and model |
| Calibration | 5 | Fit probability calibrators |
| Exploratory evaluation | 6 | Compare models and investigate business decisions |
| Final holdout | 7 | Independent one-time evaluation |

**Important:** Month 6 was used during model development. Consequently, its performance results could not serve as independent final-test evidence.

Month 7 was accessed only once after the model and policy had been frozen.

---

## 3. Frozen Fraud-Detection Model

**Selected model:** Weighted PyTorch multilayer perceptron (MLP)

**Architecture:**

- 55 input features
- First hidden layer: 64 neurons
- Second hidden layer: 32 neurons
- Output layer: 1 fraud logit
- Dropout rate: 0.10

The input pipeline started with 29 permitted raw features and produced 55 transformed features.

The following columns were excluded from the model inputs:

- `fraud_bool` — target leakage
- `month` — chronological split field
- `device_fraud_count` — previously identified leakage risk

The final candidate used sigmoid / Platt probability calibration, fitted using month-5 predictions.

No retraining or refitting occurred during the final evaluation.

---

## 4. Probability Calibration

A classification model may rank fraudulent applications effectively without producing reliable fraud probabilities.

**Discrimination:** How well the model separates fraudulent and legitimate applications.

**Calibration:** How closely predicted probabilities correspond to observed fraud frequencies.

We used Platt scaling to calibrate the weighted MLP's predictions.

Calibration quality was assessed using:

- Brier score
- Log loss
- Expected calibration error
- Risk-band diagnostics

Lower Brier scores and log loss are generally better.

Calibration is particularly important when decisions depend on an economic probability threshold.

---

## 5. Frozen Investigation Policy

The investigation policy was frozen before the final test.

| Configuration | Frozen value |
|---|---|
| Calibration | Sigmoid / Platt |
| Probability threshold | 0.0492957746 |
| Threshold as percentage | 4.9296% |
| Maximum review capacity | 5% |
| Prioritization | Descending calibrated fraud probability |
| Capacity calculation | Floor of 5% of cohort size |
| Tie-breaking | Stable input order, provisional |

An application qualifies for investigation when its calibrated fraud probability exceeds the threshold.

If more applications qualify than the available investigation capacity, the highest-risk applications are prioritized.

The 5% capacity applies to the complete evaluation cohort, not to each processing batch.

---

## 6. Business-Cost Assumptions

These assumptions are hypothetical and are not validated bank operating costs.

| Business assumption | Value |
|---|---:|
| Investigation cost | $25 per application |
| Additional false-positive cost | $10 |
| Fraud loss per case | $1,000 |
| Fraud prevention rate | 70% |

### Economic Threshold

The cost-based threshold was:

\[
\text{Threshold} =
\frac{25+10}{(0.70 \times 1000)+10}
\]

\[
\text{Threshold} \approx 0.0492958 = 4.9296\%
\]

This is a simplified economic decision rule using our defined assumptions. It is not a regulatory or universally optimal fraud threshold.

---

## 7. Independent Final Evaluation

The frozen model and policy were evaluated once on month 7.

### Month-6 vs. Month-7 Performance

| Metric | Month 6 | Month 7 |
|---|---:|---:|
| Applications | 108,168 | 96,843 |
| Confirmed fraud cases | 1,450 | 1,428 |
| Investigations | 5,408 | 4,842 |
| Fraud caught | 705 | 731 |
| Recall | 48.62% | 51.19% |
| Investigation precision | 13.04% | 15.10% |
| Average precision / PR-AUC | 0.163322 | 0.191745 |
| Brier score | 0.012077 | 0.013157 |
| Hypothetical net savings | $311,270 | $349,540 |

### Additional Month-7 Metrics

| Metric | Value |
|---|---:|
| Fraud prevalence | 1.4746% |
| ROC-AUC | 0.886352 |
| Log loss | 0.058571 |
| Expected calibration error, 10 bins | 0.004935 |
| True positives | 731 |
| False positives | 4,111 |
| False negatives | 697 |
| True negatives | 91,304 |
| False-positive rate | 4.31% |

### Interpretation

The model identified 731 of 1,428 fraud cases while investigating approximately 5% of applications.

Month-7 PR-AUC, recall, and investigation precision were higher than their month-6 development values.

However, the month-7 Brier score was also higher, indicating worse probability prediction accuracy by that metric.

These results support the model's usefulness for prioritizing fraud investigations but do not establish production readiness.

---

## 8. Final Business-Cost Calculation

### Prevented Fraud Loss

\[
731 \times \$1000 \times 0.70 = \$511,700
\]

### Investigation Cost

\[
4842 \times \$25 = \$121,050
\]

### Additional False-Positive Cost

\[
4111 \times \$10 = \$41,110
\]

### Hypothetical Net Savings

\[
\$511,700-\$121,050-\$41,110
=\boxed{\$349,540}
\]

This is the estimated improvement compared with a hypothetical no-review baseline under the frozen assumptions.

It is not verified real-world savings.

---

## 9. Reproducibility and Final-Test Safeguards

Before accessing month 7, we verified:

1. Frozen model checkpoint SHA-256.
2. Frozen probability-calibrator SHA-256.
3. Model preprocessing and inference reproducibility.
4. Previously established month-6 investigation results.
5. Final evaluation script integrity.
6. Successful development dry run.
7. Absence of a previous final-access marker.

The final evaluation created a permanent one-time access marker.

**Month 7 is now CONSUMED and must not be reused for model selection, threshold optimization, or retraining.**

The model and policy remain provisional and are not approved for production.

---

# Milestone 07 — Interview Questions and Answers

## Q1. Why did you use a chronological data split instead of a random split?

**Answer:** A chronological split better represents how a fraud model would operate in practice: training on earlier applications and evaluating on later applications. Random splitting could introduce unrealistic information sharing across time and overestimate future performance.

## Q2. Why separate model training, calibration, development evaluation, and final testing?

**Answer:** Each stage serves a different purpose. Training learns model parameters, calibration adjusts predicted probabilities, development evaluation supports analysis and model selection, and the final holdout provides independent evidence of generalization. Keeping these stages separate reduces leakage and evaluation bias.

## Q3. Why did you use a weighted MLP?

**Answer:** Fraud is a minority class, so ordinary training can favor the much larger legitimate-application class. Class weighting increases the training penalty for misclassifying fraud cases. However, it can also affect the reliability of predicted probabilities, which is one reason calibration was necessary.

## Q4. What is Platt scaling?

**Answer:** Platt scaling is a probability-calibration method that fits a sigmoid-based transformation using separate calibration data. In this project, it was fitted using month-5 model predictions and applied to the frozen MLP's outputs without changing the neural network weights.

## Q5. What is the difference between discrimination and calibration?

**Answer:** Discrimination measures how effectively a model ranks or separates positive and negative examples. Calibration measures whether predicted probabilities agree with observed outcome frequencies. A model may have strong discrimination but poorly calibrated probabilities.

## Q6. Why was PR-AUC especially important in this project?

**Answer:** Fraud cases are relatively rare, making class imbalance a central concern. Average precision summarizes precision across recall levels and is informative about identifying the minority fraud class. ROC-AUC is also useful, but it can look strong even when precision remains limited in an imbalanced dataset.

## Q7. What does a PR-AUC of 0.191745 mean?

**Answer:** The final model achieved an average precision of approximately 0.192 across its precision-recall curve. This is substantially above the final holdout fraud prevalence of approximately 0.01475. It indicates useful ranking performance, but it does not mean the model detects 19.17% of fraud cases or has 19.17% precision at the frozen operating threshold.

## Q8. What is the Brier score?

**Answer:** The Brier score is the mean squared difference between predicted probabilities and actual binary outcomes.

\[
\text{Brier Score}=
\frac{1}{N}\sum_{i=1}^{N}(p_i-y_i)^2
\]

Lower scores indicate better probabilistic accuracy. Our final Brier score was 0.013157. It was worse than month 6's 0.012077, even though the final ranking metrics improved.

## Q9. How was the investigation threshold selected?

**Answer:** We calculated a decision threshold from hypothetical fraud loss, prevention effectiveness, investigation costs, and additional false-positive costs. That calculation produced approximately 4.9296%. We froze the threshold before accessing the final holdout.

## Q10. Why did you impose a 5% investigation capacity limit?

**Answer:** Real investigation teams have limited resources. The capacity constraint makes evaluation more operationally realistic. When more applications exceed the risk threshold than can be investigated, the policy prioritizes the highest calibrated fraud probabilities.

## Q11. What is the difference between recall and precision?

**Answer:** Recall measures the percentage of actual fraud cases selected for investigation.

\[
\text{Recall}=\frac{TP}{TP+FN}
\]

Precision measures the percentage of investigated applications that are actually fraudulent.

\[
\text{Precision}=\frac{TP}{TP+FP}
\]

On month 7, recall was 51.19% and investigation precision was 15.10%.

## Q12. Why can increasing recall create business costs?

**Answer:** Detecting more fraud often requires reviewing more applications, potentially increasing false positives. Those additional investigations cost money and may impose unnecessary burdens on legitimate applicants. A useful fraud policy must balance fraud prevention against investigation capacity and false-positive costs.

## Q13. How did you calculate hypothetical financial savings?

**Answer:** We estimated prevented fraud losses by multiplying detected fraud cases by assumed loss per case and prevention rate. We then subtracted investigation costs and additional false-positive costs.

\[
\text{Net Savings}=
\text{Prevented Loss}
-\text{Investigation Cost}
-\text{Additional FP Cost}
\]

The final estimate was $349,540 using the frozen assumptions.

## Q14. Why was month 7 evaluated only once?

**Answer:** The independent holdout is intended to measure the performance of a model and policy chosen without knowledge of that holdout's outcomes. Repeatedly evaluating alternatives on the same holdout would turn it into another development dataset, weakening the validity of our final generalization estimate.

## Q15. Does completing Milestone 07 mean the model is ready for production?

**Answer:** No. The final evaluation establishes independent performance evidence, but additional requirements remain. These include stronger operational controls, validated business assumptions, stable application identifiers, API security, monitoring, deployment testing, privacy safeguards, and production approval. The current model and policy remain provisional.

---

## Final Milestone 07 Conclusion

The project completed a reproducible, one-time independent evaluation of a frozen PyTorch fraud-detection model.

The final model achieved:

- PR-AUC: **0.191745**
- ROC-AUC: **0.886352**
- Fraud recall: **51.19%**
- Investigation precision: **15.10%**
- Maximum investigation capacity: **5%**
- Hypothetical net savings: **$349,540**

The evaluation supports the model's potential value for prioritizing fraud investigations, while also identifying calibration, false-positive, and operational limitations.

**Final status:** The independent evaluation is complete. The policy is not production-approved. Month 7 has been consumed once and must not be reused for model selection.
